# Getting set up

## Setting up the notebook

In [1]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Get:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,052 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,702 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/multiverse amd64 Packages [67.4 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [2,177 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:1

## Use LLM to generate Verilog code from natural language description

### Example 1: binary_to_bcd

In [12]:
! mkdir -p binary_to_bcd
! cd binary_to_bcd && curl -O https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/binary_to_bcd_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1078  100  1078    0     0   6762      0 --:--:-- --:--:-- --:--:--  6737


In [13]:
verilog_generation_prompt = '''
I am trying to create a Verilog model binary_to_bcd_converter for a binary to binary-coded-decimal converter. It must meet the following specifications:
	- Inputs:
		- Binary input (5-bits)
	- Outputs:
		- BCD (8-bits: 4-bits for the 10's place and 4-bits for the 1's place)

How would I write a design that meets these specifications?
'''

Now you can use the openai library to generate text from your prompt.

Before using LLMs for Verilog generation, you have to first get the api-key:

openai llms: https://platform.openai.com/api-keys
deepseek llms: https://platform.deepseek.com/sign_in
free llms provided by NVIDIA: https://build.nvidia.com/explore/discover

In [19]:
from openai import OpenAI
import os

os.environ["OPENAI_BASE_URL"] = "https://integrate.api.nvidia.com/v1"  # NVIDIA API for all clients below

client = OpenAI(
    api_key = ""
)

completion = client.chat.completions.create(
  model = "nvidia/nemotron-3-super-120b-a12b",
  messages=[{"role":"user","content":verilog_generation_prompt}],
  max_tokens=4096, # limits the maximum number of tokens (words or pieces of words) that the model will generate in its response.
  stream=False
)

print(completion.choices[0].message.content)

Below is a compact, synthesizable Verilog implementation of a **5‑bit binary → 8‑bit BCD** converter.  
The module receives a 5‑bit unsigned binary number (`bin_in`) and produces an 8‑bit BCD vector (`bcd_out`) where the upper nibble holds the tens digit and the lower nibble holds the ones digit.

---

## 1. Design description

* **Range of input** – 5‑bit unsigned → 0 … 31 (decimal)  
* **Maximum BCD value** – 31₁₀ = “0011 0001”₂ → tens = 3, ones = 1  
* **Algorithm** – simple integer division / remainder:  

```
tens  = bin_in / 10;
ones  = bin_in % 10;
bcd_out = {tens, ones};   // 4‑bit tens, 4‑bit ones
```

Verilog’s `/` and `%` operators are synthesizable for constant‑width operands (the tool will infer a divider/combinatorial logic if needed). For such a small range the area impact is negligible, and the code stays easy to read and maintain.

---

## 2. Module source (`binary_to_bcd_converter.v`)

```verilog
`timescale 1ns/1ps
//===================================================

Next, extract the generated verilog code from LLM response.

In [20]:
output_verilog_code = '''
module binary_to_bcd_converter (
    input  wire [4:0] bin_in,   // 5‑bit binary input (0‑31)
    output wire [7:0] bcd_out   // 8‑bit BCD: [7:4] tens, [3:0] ones
);

    // Internal wires for the decimal digits
    wire [3:0] tens;   // 0‑3 fits in 4 bits
    wire [3:0] ones;   // 0‑9 fits in 4 bits

    // Compute tens and ones using integer division / remainder
    assign tens  = bin_in / 5'd10;   // integer division
    assign ones  = bin_in % 5'd10;   // remainder (modulo)

    // Pack the two nibbles into the 8‑bit BCD output
    assign bcd_out = {tens, ones};

endmodule
'''
filename = "binary_to_bcd/binary_to_bcd.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

Use iverilog to verify the correctness of LLM generated verilog

In [21]:
!cd binary_to_bcd/ && iverilog -g2012 -o binary_to_bcd.vvp binary_to_bcd.v binary_to_bcd_tb.v && vvp binary_to_bcd.vvp

binary_to_bcd_tb.v:8: error: port ``binary_input'' is not a port of uut.
binary_to_bcd_tb.v:8: error: port ``bcd_output'' is not a port of uut.
2 error(s) during elaboration.


In [22]:
feedback_prompt = '''
Simulating your case-lookup design with my testbench gives:
binary_to_bcd_tb.v:8: error: port ``binary_input'' is not a port of uut.
binary_to_bcd_tb.v:8: error: port ``bcd_output'' is not a port of uut.
Please rename the ports to binary_input [4:0] and bcd_output [7:0]. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
module binary_to_bcd_converter (
    input  wire [4:0] binary_input,   // 5‑bit binary input
    output wire [7:0] bcd_output      // 8‑bit BCD: [7:4] tens, [3:0] ones
);

    // Internal decimal digits
    wire [3:0] tens;
    wire [3:0] ones;

    // Compute tens and ones using division / remainder
    assign tens  = binary_input / 5'd10;
    assign ones  = binary_input % 5'd10;

    // Pack the two nibbles into the BCD output
    assign bcd_output = {tens, ones};

endmodule
```


In [23]:
output_verilog_code = '''
module binary_to_bcd_converter (
    input  wire [4:0] binary_input,   // 5‑bit binary input
    output wire [7:0] bcd_output      // 8‑bit BCD: [7:4] tens, [3:0] ones
);

    // Internal decimal digits
    wire [3:0] tens;
    wire [3:0] ones;

    // Compute tens and ones using division / remainder
    assign tens  = binary_input / 5'd10;
    assign ones  = binary_input % 5'd10;

    // Pack the two nibbles into the BCD output
    assign bcd_output = {tens, ones};

endmodule
'''
filename = "binary_to_bcd/binary_to_bcd.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [24]:
!cd binary_to_bcd/ && iverilog -g2012 -o binary_to_bcd.vvp binary_to_bcd.v binary_to_bcd_tb.v && vvp binary_to_bcd.vvp

Testing Binary-to-BCD Converter...
VCD info: dumpfile my_design.vcd opened for output.
All test cases passed!
binary_to_bcd_tb.v:38: $finish called at 320000 (1ps)


### Example 2: Sequence detector

In [25]:
! mkdir -p sequence_detector
! cd sequence_detector && curl -O https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2114  100  2114    0     0  14774      0 --:--:-- --:--:-- --:--:-- 14887


In [37]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a sequence detector. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Data (3 bits)
	- Outputs:
		- Sequence found

While enabled, it should detect the following sequence of binary input values:
	- 0b001
	- 0b101
	- 0b110
	- 0b000
	- 0b110
	- 0b110
	- 0b011
	- 0b101

How would I write a design that meets these specifications?
'''

In [38]:
from openai import OpenAI
import os
client = OpenAI(
    api_key = ""
)

completion = client.chat.completions.create(
  model = "nvidia/nemotron-3-super-120b-a12b",
  messages=[{"role":"user","content":verilog_generation_prompt}],
  max_tokens=4096, # limits the maximum number of tokens (words or pieces of words) that the model will generate in its response.
  stream=False
)

print(completion.choices[0].message.content)

Below is a **self‑contained, synthesizable Verilog model** that implements the 8‑step sequence detector you described.

* **Inputs**  
  * `clk`          – system clock (posedge‑triggered)  
  * `reset_n`      – **active‑low** asynchronous reset (`0` = reset)  
  * `data[2:0]`    – 3‑bit sample that arrives every clock cycle  

* **Output**  
  * `seq_found`    – goes high for **one clock cycle** when the complete 8‑word pattern has been seen  

The detector is implemented as a simple Moore‑type FSM.  
If a mismatch occurs the FSM restarts looking for the first word (`001`).  
(You can easily change the restart policy to allow overlapping detection – see the comment in the code.)

---

## 1. FSM Description  

| State | Meaning (what has been matched so far) | Expected next 3‑bit word |
|-------|----------------------------------------|--------------------------|
| **S0**| idle – waiting for the first word      | `3'b001`                 |
| **S1**| …001                                

Next, extract the generated verilog code from LLM response.

In [39]:
output_verilog_code = '''
module seq_detector #(
    parameter DATA_W = 3               // width of the data bus
) (
    input  wire               clk,
    input  wire               reset_n,   // active‑low asynchronous reset
    input  wire [DATA_W-1:0] data,
    output reg                seq_found
);

    // -------------------------------------------------
    // FSM state encoding (one‑hot is also fine, but binary is compact)
    // -------------------------------------------------
    typedef enum logic [3:0] {
        S0 = 4'd0,
        S1 = 4'd1,
        S2 = 4'd2,
        S3 = 4'd3,
        S4 = 4'd4,
        S5 = 4'd5,
        S6 = 4'd6,
        S7 = 4'd7,
        S8 = 4'd8   // detection state
    } state_t;

    state_t state_q, state_d;

    // -------------------------------------------------
    // Asynchronous reset (active low) + synchronous state update
    // -------------------------------------------------
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            state_q <= S0;
            seq_found <= 1'b0;
        end else begin
            state_q   <= state_d;
            seq_found <= (state_q == S8);   // pulse for one cycle when in S8
        end
    end

    // -------------------------------------------------
    // Next‑state logic
    // -------------------------------------------------
    always @* begin
        // Default: stay in current state (will be overwritten by match/mismatch)
        state_d = state_q;

        unique case (state_q)
            S0: begin
                if (data == 3'b001) state_d = S1;
                else                state_d = S0;   // stay looking for first word
            end
            S1: begin
                if (data == 3'b101) state_d = S2;
                else                state_d = (data == 3'b001) ? S1 : S0;
            end
            S2: begin
                if (data == 3'b110) state_d = S3;
                else                state_d = (data == 3'b001) ? S1 : S0;
            end
            S3: begin
                if (data == 3'b000) state_d = S4;
                else                state_d = (data == 3'b001) ? S1 : S0;
            end
            S4: begin
                if (data == 3'b110) state_d = S5;
                else                state_d = (data == 3'b001) ? S1 : S0;
            end
            S5: begin
                if (data == 3'b110) state_d = S6;
                else                state_d = (data == 3'b001) ? S1 : S0;
            end
            S6: begin
                if (data == 3'b011) state_d = S7;
                else                state_d = (data == 3'b001) ? S1 : S0;
            end
            S7: begin
                if (data == 3'b101) state_d = S8;   // full match
                else                state_d = (data == 3'b001) ? S1 : S0;
            end
            S8: begin   // detection state – go back to idle for next search
                state_d = S0;
            end
        endcase
    end

endmodule
'''
filename = "sequence_detector/sequence_detector.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [40]:
!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

sequence_detector_tb.v:10: error: Unknown module type: sequence_detector
2 error(s) during elaboration.
*** These modules were missing:
        sequence_detector referenced 1 times.
***


Feedback round 1: give the testbench error back to the LLM.

In [41]:
feedback_prompt = '''
Simulating with my testbench gives:
error: Unknown module type: sequence_detector
The testbench instantiates sequence_detector with ports clk, reset_n, data [2:0], sequence_found. Please use these names, and use plain Verilog (no typedef enum). Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
`timescale 1ns/1ps
module sequence_detector (
    input  wire        clk,
    input  wire        reset_n,   // active‑low asynchronous reset
    input  wire [2:0]  data,
    output reg         sequence_found
);

    // ----- state encoding (binary, no typedef) -----
    parameter S0 = 4'd0;
    parameter S1 = 4'd1;
    parameter S2 = 4'd2;
    parameter S3 = 4'd3;
    parameter S4 = 4'd4;
    parameter S5 = 4'd5;
    parameter S6 = 4'd6;
    parameter S7 = 4'd7;
    parameter S8 = 4'd8;   // detection state

    reg [3:0] state;
    reg [3:0] next_state;

    // ----- state register (async reset) -----
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n)
            state <= S0;
        else
            state <= next_state;
    end

    // ----- output: pulse for one clock when in S8 -----
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n)
            sequence_found <= 1'b0;
        else
            sequence_found <= (state == S8);
 

Next, extract the corrected verilog code from LLM response.

In [45]:
output_verilog_code = '''
module sequence_detector (
    input  wire        clk,
    input  wire        reset_n,   // active‑low asynchronous reset
    input  wire [2:0]  data,
    output reg         sequence_found
);

    // ----- state encoding (binary, no typedef) -----
    parameter S0 = 4'd0;
    parameter S1 = 4'd1;
    parameter S2 = 4'd2;
    parameter S3 = 4'd3;
    parameter S4 = 4'd4;
    parameter S5 = 4'd5;
    parameter S6 = 4'd6;
    parameter S7 = 4'd7;
    parameter S8 = 4'd8;   // detection state

    reg [3:0] state;
    reg [3:0] next_state;

    // ----- state register (async reset) -----
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n)
            state <= S0;
        else
            state <= next_state;
    end

    // ----- output: pulse for one clock when in S8 -----
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n)
            sequence_found <= 1'b0;
        else
            sequence_found <= (state == S8);
    end

    // ----- next‑state logic -----
    always @* begin
        // default: stay in current state (will be overridden on match/mismatch)
        next_state = state;

        case (state)
            S0: begin
                if (data == 3'b001) next_state = S1;
                else                next_state = S0;
            end
            S1: begin
                if (data == 3'b101) next_state = S2;
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S2: begin
                if (data == 3'b110) next_state = S3;
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S3: begin
                if (data == 3'b000) next_state = S4;
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S4: begin
                if (data == 3'b110) next_state = S5;
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S5: begin
                if (data == 3'b110) next_state = S6;
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S6: begin
                if (data == 3'b011) next_state = S7;
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S7: begin
                if (data == 3'b101) next_state = S8;   // full pattern seen
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S8: begin   // detection state – go back to idle for next search
                next_state = S0;
            end
            default: next_state = S0;
        endcase
    end

endmodule
'''


filename = "sequence_detector/sequence_detector.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [46]:
!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

Error: Cycle 8, Expected: 1, Got: 0
sequence_detector_tb.v:39: $finish called at 85000 (1ps)


In [50]:
feedback_prompt = '''
Now the testbench reports: "Error: Cycle 8, Expected: 1, Got: 0".
The testbench expects sequence_found to be high in the same cycle the final 0b101 is applied. Please fix the design. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
`timescale 1ns/1ps
module sequence_detector (
    input  wire        clk,
    input  wire        reset_n,   // active‑low asynchronous reset
    input  wire [2:0]  data,
    output reg         sequence_found
);

    // ----- state encoding -----
    parameter S0 = 4'd0;
    parameter S1 = 4'd1;
    parameter S2 = 4'd2;
    parameter S3 = 4'd3;
    parameter S4 = 4'd4;
    parameter S5 = 4'd5;
    parameter S6 = 4'd6;
    parameter S7 = 4'd7;

    reg [3:0] state;
    reg [3:0] next_state;

    // ----- state register (async reset) -----
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            state <= S0;
        end else begin
            state <= next_state;
        end
    end

    // ----- next‑state logic -----
    always @* begin
        // default: stay in current state (will be overridden on match/mismatch)
        next_state = state;

        case (state)
            S0: begin
                if (data == 3'b001) next_state = S1;
   

In [61]:
output_verilog_code = '''
module sequence_detector (
    input  wire        clk,
    input  wire        reset_n,   // active‑low asynchronous reset
    input  wire [2:0]  data,
    output reg         sequence_found
);

    // ----- state encoding -----
    parameter S0 = 4'd0;
    parameter S1 = 4'd1;
    parameter S2 = 4'd2;
    parameter S3 = 4'd3;
    parameter S4 = 4'd4;
    parameter S5 = 4'd5;
    parameter S6 = 4'd6;
    parameter S7 = 4'd7;

    reg [3:0] state;
    reg [3:0] next_state;

    // ----- state register (async reset) -----
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            state <= S0;
        end else begin
            state <= next_state;
        end
    end

    // ----- next‑state logic -----
    always @* begin
        // default: stay in current state (will be overridden on match/mismatch)
        next_state = state;

        case (state)
            S0: begin
                if (data == 3'b001) next_state = S1;
                else                next_state = S0;
            end
            S1: begin
                if (data == 3'b101) next_state = S2;
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S2: begin
                if (data == 3'b110) next_state = S3;
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S3: begin
                if (data == 3'b000) next_state = S4;
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S4: begin
                if (data == 3'b110) next_state = S5;
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S5: begin
                if (data == 3'b110) next_state = S6;
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S6: begin
                if (data == 3'b011) next_state = S7;
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            S7: begin
                if (data == 3'b101) next_state = S0;   // pattern completed -> restart
                else                next_state = (data == 3'b001) ? S1 : S0;
            end
            default: next_state = S0;
        endcase
    end

    // ----- output logic (combinational) -----
    always @* begin
        if (!reset_n) begin
            sequence_found = 1'b0;
        end else begin
            sequence_found = (state == S7 && data == 3'b101);
        end
    end

endmodule
'''


filename = "sequence_detector/sequence_detector.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [62]:
!cd sequence_detector/ && iverilog -g2012 -o sequence_detector.vvp sequence_detector.v sequence_detector_tb.v && vvp sequence_detector.vvp

All test cases passed.
sequence_detector_tb.v:73: $finish called at 125000 (1ps)


### Example 3: Shift Register

In [53]:
! mkdir -p shift_register
! cd shift_register && curl -O https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/shift_register_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1367  100  1367    0     0   7071      0 --:--:-- --:--:-- --:--:--  7046


In [54]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a shift register. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Data (1 bit)
		- Shift enable
	- Outputs:
		- Data (8 bits)

How would I write a design that meets these specifications?
'''

In [55]:
from openai import OpenAI

client = OpenAI(
  api_key = ""
)

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=[{"role":"user","content":verilog_generation_prompt}],
  max_tokens=4096, # limits the maximum number of tokens (words or pieces of words) that the model will generate in its response.
  stream=False
)

print(completion.choices[0].message.content)



Below is a self‑contained, synthesizable Verilog implementation that matches the exact interface you described:

* **Inputs** – `clk` (clock), `rst_n` (active‑low reset), `din` (1‑bit serial data), `shift_en` (shift enable)  
* **Output** – `q` (8‑bit parallel data)  

The register shifts left on every clock cycle when `shift_en` is asserted, inserting the new bit (`din`) into the least‑significant position. When `shift_en` is low the register holds its current value. An active‑low reset forces all bits to `0`.

```verilog
`timescale 1ns/1ps

/* 8‑bit shift register with synchronous active‑low reset
 *
 *  Port list:
 *    input  wire clk        ; // system clock
 *    input  wire rst_n      ; // active‑low reset (0 = reset)
 *    input  wire din        ; // serial data input (1 bit)
 *    input  wire shift_en   ; // shift enable (high = shift)
 *    output reg [7:0] q     ; // parallel output (8 bits)
 */
module shift_reg_8bit (
    input  wire       clk,
    input  wire       rst_n,


In [56]:
output_verilog_code = '''
module shift_reg_8bit (
    input  wire       clk,
    input  wire       rst_n,
    input  wire       din,
    input  wire       shift_en,
    output reg [7:0]  q
);

    // Synchronous reset – active low
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin               // reset asserted (active‑low)
            q <= 8'b0;
        end else if (shift_en) begin    // shift enabled
            // Left‑shift: move bits toward MSB, insert din at LSB
            q <= {q[6:0], din};
        end else begin                  // hold current value
            q <= q;
        end
    end

endmodule
'''
filename = "shift_register/shift_register.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [57]:
!cd shift_register/ && iverilog -o shift_register.vvp shift_register.v shift_register_tb.v && vvp shift_register.vvp

shift_register_tb.v:11: error: Unknown module type: shift_register
2 error(s) during elaboration.
*** These modules were missing:
        shift_register referenced 1 times.
***


In [58]:
feedback_prompt = '''
My testbench instantiates the module as shift_register with ports clk, reset_n, data_in, shift_enable, data_out [7:0]. Please rename the module and ports to match. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
`timescale 1ns/1ps

module shift_register (
    input  wire       clk,
    input  wire       reset_n,
    input  wire       data_in,
    input  wire       shift_enable,
    output reg [7:0] data_out
);

    // Synchronous active‑low reset
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin               // reset asserted (active‑low)
            data_out <= 8'b0;
        end else if (shift_enable) begin  // shift enabled
            // Left‑shift: insert data_in at LSB
            data_out <= {data_out[6:0], data_in};
        end else begin                    // hold current value
            data_out <= data_out;
        end
    end

endmodule
```


In [63]:
output_verilog_code = '''
module shift_register (
    input  wire       clk,
    input  wire       reset_n,
    input  wire       data_in,
    input  wire       shift_enable,
    output reg [7:0] data_out
);

    // Synchronous active‑low reset
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin               // reset asserted (active‑low)
            data_out <= 8'b0;
        end else if (shift_enable) begin  // shift enabled
            // Left‑shift: insert data_in at LSB
            data_out <= {data_out[6:0], data_in};
        end else begin                    // hold current value
            data_out <= data_out;
        end
    end

endmodule
'''


filename = "shift_register/shift_register.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [64]:
!cd shift_register/ && iverilog -o shift_register.vvp shift_register.v shift_register_tb.v && vvp shift_register.vvp

Error: Test case 0 failed. Expected: 00001010, Got: xxxxxxxx
shift_register_tb.v:47: $finish called at 10000 (1ps)


**========Note==========**

the shift register design above matches the specification, but the provided testbench cannot be passed by a correct design: it never asserts reset before the first check yet expects `00001010`, and its expected values shift right every cycle regardless of `shift_enable`. I was unsure if I could change the testbench so I left this at the correct design.

### Example 4: Abro State Machine

In [65]:
! mkdir -p abro_state_machine
!cd abro_state_machine && curl -O https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/abro_state_machine_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2762  100  2762    0     0  14110      0 --:--:-- --:--:-- --:--:-- 14164


In [66]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for an ABRO state machine. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - A
        - B
    - Outputs:
        - O
        - State

Other than the main output from ABRO machine, it should output the current state of the machine for use in verification.

The states for this state machine should be one-hot encoded.

How would I write a design that meets these specifications?
'''

In [67]:
from openai import OpenAI

client = OpenAI(
  api_key = ""
)

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=[{"role":"user","content":verilog_generation_prompt}],
  max_tokens=4096, # limits the maximum number of tokens (words or pieces of words) that the model will generate in its response.
  stream=False
)

print(completion.choices[0].message.content)

Below is a **complete, synthesizable Verilog model** of an ABRO finite‑state machine that satisfies every item in your request:

*   **One‑hot state encoding** (4‑bit vector, only one bit set at a time)  
*   **Inputs** – `clk`, active‑low reset `rst_n`, and two data inputs `A` and `B`  
*   **Outputs** – the machine output `O` (asserted only in the **O** state) and the full state vector `state` (for verification)  
*   **Reset** – asynchronously forces the machine to the **A** state (the reset state)  
*   **Clock** – synchronous state update on the rising edge of `clk`  

The transition table used here is a classic “sequence‑detector” interpretation of the name **ABRO**:

| Current state | `A` `B` | Next state | Meaning (what we have seen so far) |
|---------------|--------|------------|------------------------------------|
| **A** (0001)  | 00 → A | stay in A  | we have not yet seen a useful symbol |
|               | 01 → B | go to B    | we have seen an **A** (first symbol) |
|   

In [68]:
output_verilog_code = '''
module abro_fsm (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // data input A
    input  wire        B,        // data input B
    output reg         O,        // machine output (high in state O)
    output wire [3:0]  state     // one‑hot state vector (for verification)
);

    // --------------------------------------------------------------
    //  One‑hot state encoding
    // --------------------------------------------------------------
    localparam [3:0] S_A = 4'b0001;   // state A
    localparam [3:0] S_B = 4'b0010;   // state B
    localparam [3:0] S_R = 4'b0100;   // state R
    localparam [3:0] S_O = 4'b1000;   // state O

    // --------------------------------------------------------------
    //  State registers (asynchronous reset)
    // --------------------------------------------------------------
    reg [3:0] current_state;
    reg [3:0] next_state;

    // Synchronous state update
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            current_state <= S_A;          // reset to state A
        else
            current_state <= next_state;
    end

    // --------------------------------------------------------------
    //  Next‑state logic (combinational)
    // --------------------------------------------------------------
    always @* begin
        // Default – stay in current state (helps avoid latches)
        next_state = current_state;

        case (current_state)
            S_A: begin
                if (!A && !B) next_state = S_A;   // 00 -> stay A
                else if (!A && B)  next_state = S_B; // 01 -> B
                else if ( A && !B) next_state = S_R; // 10 -> R
                else               next_state = S_O; // 11 -> O
            end
            S_B: begin
                if (!A && !B) next_state = S_A;
                else if (!A && B)  next_state = S_B;
                else if ( A && !B) next_state = S_R;
                else               next_state = S_O;
            end
            S_R: begin
                if (!A && !B) next_state = S_A;
                else if (!A && B)  next_state = S_B;
                else if ( A && !B) next_state = S_R;
                else               next_state = S_O;
            end
            S_O: begin
                // Once we have seen the full pattern we stay in O
                // until a reset occurs.  (Change this block if you
                // want a different behaviour, e.g. go back to A.)
                if (!A && !B) next_state = S_A;
                else if (!A && B)  next_state = S_B;
                else if ( A && !B) next_state = S_R;
                else               next_state = S_O;
            end
            default: next_state = S_A;   // safety net
        endcase
    end

    // --------------------------------------------------------------
    //  Output logic
    // --------------------------------------------------------------
    // Moore output: high only when we are in state O
    always @* begin
        O = (current_state == S_O);
    end

    // If you prefer a Mealy output (high on the transition that
    // completes the sequence) uncomment the following line and
    // comment‑out the Moore assignment above:
    // assign O = (current_state == S_R) && A && B;   // O asserted when we see the 4th symbol while in R

    // --------------------------------------------------------------
    //  State vector for verification (one‑hot)
    // --------------------------------------------------------------
    assign state = current_state;

endmodule
'''
filename = "abro_state_machine/abro_state_machine.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [69]:
!cd abro_state_machine/ && iverilog -o abro_state_machine.vvp abro_state_machine.v abro_state_machine_tb.v && vvp abro_state_machine.vvp

abro_state_machine_tb.v:12: error: Unknown module type: abro_state_machine
2 error(s) during elaboration.
*** These modules were missing:
        abro_state_machine referenced 1 times.
***


In [70]:
feedback_prompt = '''
Simulating with my testbench gives:
error: Unknown module type: abro_state_machine
The testbench instantiates the module as abro_state_machine with ports clk, rst_n, A, B, O, State [3:0] (capital S). Please rename the module and ports to match. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
`timescale 1ns/1ps

module abro_state_machine (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // data input A
    input  wire        B,        // data input B
    output reg         O,        // machine output (high in state O)
    output wire [3:0]  State     // one‑hot state vector (for verification)
);

    // --------------------------------------------------------------
    //  One‑hot state encoding
    // --------------------------------------------------------------
    localparam [3:0] S_A = 4'b0001;   // state A
    localparam [3:0] S_B = 4'b0010;   // state B
    localparam [3:0] S_R = 4'b0100;   // state R
    localparam [3:0] S_O = 4'b1000;   // state O

    // --------------------------------------------------------------
    //  State registers (asynchronous reset)
    // --------------------------------------------------------------
    reg [3:0] current_stat

In [71]:
output_verilog_code = '''
module abro_state_machine (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // data input A
    input  wire        B,        // data input B
    output reg         O,        // machine output (high in state O)
    output wire [3:0]  State     // one‑hot state vector (for verification)
);

    // --------------------------------------------------------------
    //  One‑hot state encoding
    // --------------------------------------------------------------
    localparam [3:0] S_A = 4'b0001;   // state A
    localparam [3:0] S_B = 4'b0010;   // state B
    localparam [3:0] S_R = 4'b0100;   // state R
    localparam [3:0] S_O = 4'b1000;   // state O

    // --------------------------------------------------------------
    //  State registers (asynchronous reset)
    // --------------------------------------------------------------
    reg [3:0] current_state;
    reg [3:0] next_state;

    // Synchronous state update
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            current_state <= S_A;          // reset to state A
        else
            current_state <= next_state;
    end

    // --------------------------------------------------------------
    //  Next‑state logic (combinational)
    // --------------------------------------------------------------
    always @* begin
        // Default – stay in current state (helps avoid latches)
        next_state = current_state;

        case (current_state)
            S_A: begin
                if (!A && !B) next_state = S_A;   // 00 -> stay A
                else if (!A && B)  next_state = S_B; // 01 -> B
                else if ( A && !B) next_state = S_R; // 10 -> R
                else               next_state = S_O; // 11 -> O
            end
            S_B: begin
                if (!A && !B) next_state = S_A;
                else if (!A && B)  next_state = S_B;
                else if ( A && !B) next_state = S_R;
                else               next_state = S_O;
            end
            S_R: begin
                if (!A && !B) next_state = S_A;
                else if (!A && B)  next_state = S_B;
                else if ( A && !B) next_state = S_R;
                else               next_state = S_O;
            end
            S_O: begin
                // Remain in O until a reset (you may change this if desired)
                if (!A && !B) next_state = S_A;
                else if (!A && B)  next_state = S_B;
                else if ( A && !B) next_state = S_R;
                else               next_state = S_O;
            end
            default: next_state = S_A;   // safety net
        endcase
    end

    // --------------------------------------------------------------
    //  Output logic (Moore: O asserted only in state O)
    // --------------------------------------------------------------
    always @* begin
        O = (current_state == S_O);
    end

    // --------------------------------------------------------------
    //  State vector for verification (one‑hot)
    // --------------------------------------------------------------
    assign State = current_state;

endmodule
'''
filename = "abro_state_machine/abro_state_machine.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [72]:
!cd abro_state_machine/ && iverilog -o abro_state_machine.vvp abro_state_machine.v abro_state_machine_tb.v && vvp abro_state_machine.vvp

VCD info: dumpfile my_design.vcd opened for output.
Error: Test case 3 failed (A=1, B=1)
  Expected: O=1, State=0100
  Got: O=0, State=0100
abro_state_machine_tb.v:85: $finish called at 45000 (1ps)


In [73]:
feedback_prompt = '''
Now the testbench reports:
Error: Test case 3 failed (A=1, B=1)
  Expected: O=1, State=0100
  Got: O=0, State=0100
O should be high when both A and B have been seen (state 0100), not in a separate fourth state. Please fix the design. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
`timescale 1ns/1ps

module abro_state_machine (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // data input A
    input  wire        B,        // data input B
    output reg         O,        // asserted when both A and B have been seen
    output wire [3:0]  State     // one‑hot state vector (for verification)
);

    // --------------------------------------------------------------
    //  One‑hot state encoding
    // --------------------------------------------------------------
    localparam [3:0] S_A = 4'b0001;   // state A  (seen A only)
    localparam [3:0] S_B = 4'b0010;   // state B  (seen B only)
    localparam [3:0] S_R = 4'b0100;   // state R  (seen both A and B) -> O asserted
    localparam [3:0] S_O = 4'b1000;   // unused state (kept for 4‑bit one‑hot)

    // --------------------------------------------------------------
    //  State registers (asynchronous

In [74]:
output_verilog_code = '''
module abro_state_machine (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // data input A
    input  wire        B,        // data input B
    output reg         O,        // asserted when both A and B have been seen
    output wire [3:0]  State     // one‑hot state vector (for verification)
);

    // --------------------------------------------------------------
    //  One‑hot state encoding
    // --------------------------------------------------------------
    localparam [3:0] S_A = 4'b0001;   // state A  (seen A only)
    localparam [3:0] S_B = 4'b0010;   // state B  (seen B only)
    localparam [3:0] S_R = 4'b0100;   // state R  (seen both A and B) -> O asserted
    localparam [3:0] S_O = 4'b1000;   // unused state (kept for 4‑bit one‑hot)

    // --------------------------------------------------------------
    //  State registers (asynchronous reset)
    // --------------------------------------------------------------
    reg [3:0] current_state;
    reg [3:0] next_state;

    // Synchronous state update
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            current_state <= S_A;          // reset to state A
        else
            current_state <= next_state;
    end

    // --------------------------------------------------------------
    //  Next‑state logic: go to R when both inputs are 1, otherwise hold
    // --------------------------------------------------------------
    always @* begin
        if (A & B)
            next_state = S_R;              // both A and B seen → state R
        else
            next_state = current_state;    // hold current state
    end

    // --------------------------------------------------------------
    //  Output logic: O asserted only in state R (both seen)
    // --------------------------------------------------------------
    always @* begin
        O = (current_state == S_R);
    end

    // --------------------------------------------------------------
    //  State vector for verification (one‑hot)
    // --------------------------------------------------------------
    assign State = current_state;

endmodule
'''
filename = "abro_state_machine/abro_state_machine.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [75]:
!cd abro_state_machine/ && iverilog -o abro_state_machine.vvp abro_state_machine.v abro_state_machine_tb.v && vvp abro_state_machine.vvp

VCD info: dumpfile my_design.vcd opened for output.
Error: Test case 2 failed (A=1, B=0)
  Expected: O=0, State=0010
  Got: O=0, State=0001
abro_state_machine_tb.v:85: $finish called at 35000 (1ps)


In [76]:
feedback_prompt = '''
That fixed O, but now an earlier test fails:
Error: Test case 2 failed (A=1, B=0)
  Expected: O=0, State=0010
  Got: O=0, State=0001
You removed the other transitions. The state should still change when only A or only B is high, as in your previous version; only the A=1, B=1 case should go to 0100 with O high. Please fix the design. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
`timescale 1ns/1ps

module abro_state_machine (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // data input A
    input  wire        B,        // data input B
    output reg         O,        // high when both A and B have been seen
    output wire [3:0]  State     // one‑hot state vector (for verification)
);

    // --------------------------------------------------------------
    //  One‑hot state encoding
    // --------------------------------------------------------------
    localparam [3:0] S_NONE = 4'b0001; // reset state: nothing seen yet
    localparam [3:0] S_A    = 4'b0010; // A has been seen (B not yet)
    localparam [3:0] S_B    = 4'b0100; // B has been seen (A not yet)
    localparam [3:0] S_AB   = 4'b1000; // both A and B have been seen -> O asserted

    // --------------------------------------------------------------
    //  State registers (asynchronou

In [77]:
output_verilog_code = '''
module abro_state_machine (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // data input A
    input  wire        B,        // data input B
    output reg         O,        // high when both A and B have been seen
    output wire [3:0]  State     // one‑hot state vector (for verification)
);

    // --------------------------------------------------------------
    //  One‑hot state encoding
    // --------------------------------------------------------------
    localparam [3:0] S_NONE = 4'b0001; // reset state: nothing seen yet
    localparam [3:0] S_A    = 4'b0010; // A has been seen (B not yet)
    localparam [3:0] S_B    = 4'b0100; // B has been seen (A not yet)
    localparam [3:0] S_AB   = 4'b1000; // both A and B have been seen -> O asserted

    // --------------------------------------------------------------
    //  State registers (asynchronous reset)
    // --------------------------------------------------------------
    reg [3:0] current_state;
    reg [3:0] next_state;

    // Synchronous state update
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            current_state <= S_NONE;
        else
            current_state <= next_state;
    end

    // --------------------------------------------------------------
    //  Next‑state logic
    // --------------------------------------------------------------
    always @* begin
        case (current_state)
            S_NONE: begin
                if (A & ~B) next_state = S_A;          // only A high
                else if (~A & B) next_state = S_B;      // only B high
                else if (A & B) next_state = S_AB;      // both high
                else          next_state = S_NONE;      // stay in reset state
            end
            S_A: begin
                // Once A has been seen, stay in S_A until B arrives
                if (B) next_state = S_AB;
                else   next_state = S_A;
            end
            S_B: begin
                // Once B has been seen, stay in S_B until A arrives
                if (A) next_state = S_AB;
                else   next_state = S_B;
            end
            S_AB: begin
                // Absorbing state – both have been seen
                next_state = S_AB;
            end
            default: next_state = S_NONE; // safety fallback
        endcase
    end

    // --------------------------------------------------------------
    //  Output logic
    // --------------------------------------------------------------
    always @* begin
        O = (current_state == S_AB);
    end

    // --------------------------------------------------------------
    //  State vector for verification (one‑hot)
    // --------------------------------------------------------------
    assign State = current_state;

endmodule
'''

filename = "abro_state_machine/abro_state_machine.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [78]:
!cd abro_state_machine/ && iverilog -o abro_state_machine.vvp abro_state_machine.v abro_state_machine_tb.v && vvp abro_state_machine.vvp

VCD info: dumpfile my_design.vcd opened for output.
Error: Test case 2 failed (A=1, B=0)
  Expected: O=0, State=0010
  Got: O=0, State=0100
abro_state_machine_tb.v:85: $finish called at 35000 (1ps)


In [79]:
feedback_prompt = '''
Still failing:
Error: Test case 2 failed (A=1, B=0)
  Expected: O=0, State=0010
  Got: O=0, State=0100
My testbench expects this behaviour on each clock edge (3 one-hot states only, no 1000 state):
	- A=1, B=1 -> State 0100, O=1
	- A=1, B=0 -> State 0010, O=0
	- A=0, B=1 -> State 0001, O=0 (B alone returns to the reset state)
	- A=0, B=0 -> hold State
O is registered (updated on the clock edge). Reset gives State 0001, O=0. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
`timescale 1ns/1ps

module abro_state_machine (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // data input A
    input  wire        B,        // data input B
    output reg         O,        // registered output (high when both A and B have been seen)
    output reg [2:0]   State     // one‑hot state vector (for verification)
);

    // --------------------------------------------------------------
    //  One‑hot state encoding (3 bits)
    // --------------------------------------------------------------
    localparam [2:0] S_RESET = 3'b001; // reset state
    localparam [2:0] S_A     = 3'b010; // A seen (B not yet)
    localparam [2:0] S_AB    = 3'b100; // both A and B have been seen → O asserted

    // --------------------------------------------------------------
    //  State registers (asynchronous reset)
    // -----------------------------------------------------

In [80]:
output_verilog_code = '''
module abro_state_machine (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // data input A
    input  wire        B,        // data input B
    output reg         O,        // registered output (high when both A and B have been seen)
    output reg [2:0]   State     // one‑hot state vector (for verification)
);

    // --------------------------------------------------------------
    //  One‑hot state encoding (3 bits)
    // --------------------------------------------------------------
    localparam [2:0] S_RESET = 3'b001; // reset state
    localparam [2:0] S_A     = 3'b010; // A seen (B not yet)
    localparam [2:0] S_AB    = 3'b100; // both A and B have been seen → O asserted

    // --------------------------------------------------------------
    //  State registers (asynchronous reset)
    // --------------------------------------------------------------
    reg [2:0] current_state;
    reg [2:0] next_state;

    // Synchronous state update
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            current_state <= S_RESET;
        else
            current_state <= next_state;
    end

    // --------------------------------------------------------------
    //  Next‑state logic (combinational)
    // --------------------------------------------------------------
    always @* begin
        case (current_state)
            S_RESET: begin
                // A alone → S_A
                if (A & ~B)  next_state = S_A;
                // A & B together → S_AB (output asserted)
                else if (A & B) next_state = S_AB;
                // B alone or both 0 → stay in reset
                else            next_state = S_RESET;
            end
            S_A: begin
                // Once A has been seen, wait for B
                if (B) next_state = S_AB;   // B arrived → both seen
                else   next_state = S_A;    // hold A‑only state
            end
            S_AB: begin
                // Absorbing state – both have been seen
                next_state = S_AB;
            end
            default: next_state = S_RESET; // safety fallback
        endcase
    end

    // --------------------------------------------------------------
    //  Output logic – O is registered (updated on clock edge)
    // --------------------------------------------------------------
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            O <= 1'b0;
        else
            O <= (next_state == S_AB);   // O reflects the next state after the clock edge
    end

    // --------------------------------------------------------------
    //  State vector for verification (one‑hot)
    // --------------------------------------------------------------
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n)
            State <= S_RESET;
        else
            State <= next_state;
    end

endmodule
'''

filename = "abro_state_machine/abro_state_machine.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [81]:
!cd abro_state_machine/ && iverilog -o abro_state_machine.vvp abro_state_machine.v abro_state_machine_tb.v && vvp abro_state_machine.vvp

abro_state_machine_tb.v:12: warning: Port 6 (State) of abro_state_machine expects 3 bits, got 4.
abro_state_machine_tb.v:12:        : Padding 1 high bits of the expression.
VCD info: dumpfile my_design.vcd opened for output.
Error: Test case 2 failed (A=1, B=0)
  Expected: O=0, State=0010
  Got: O=0, State=0001
abro_state_machine_tb.v:85: $finish called at 35000 (1ps)


In [82]:
feedback_prompt = '''
Still failing:
Error: Test case 2 failed (A=1, B=0)
  Expected: O=0, State=0010
  Got: O=0, State=0001
Also iverilog warns: "Port 6 (State) of abro_state_machine expects 3 bits, got 4." State must be [3:0].
The transitions in my table apply from every state, not only the reset state.
My testbench changes A and B at the same time as the clock edge, so please compute the new State and O directly from A and B inside the clocked always block, without a separate combinational next_state block. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
`timescale 1ns/1ps

module abro_state_machine (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // data input A
    input  wire        B,        // data input B
    output reg         O,        // registered output (high when both A and B have been seen)
    output reg [3:0]   State     // one‑hot state vector (for verification)
);

    // one‑hot encoding (4 bits – one unused code is allowed)
    localparam [3:0] S_RESET = 4'b0001; // no input seen yet
    localparam [3:0] S_A     = 4'b0010; // A has been seen (B not yet)
    localparam [3:0] S_AB    = 4'b0100; // both A and B have been seen

    // state register
    reg [3:0] state_reg;

    // --------------------------------------------------------------
    //  Sequential logic – computes next state and registered output
    // --------------------------------------------------------------
    always @(posedge clk or neg

In [83]:
output_verilog_code = '''
module abro_state_machine (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // data input A
    input  wire        B,        // data input B
    output reg         O,        // registered output (high when both A and B have been seen)
    output reg [3:0]   State     // one‑hot state vector (for verification)
);

    // one‑hot encoding (4 bits – one unused code is allowed)
    localparam [3:0] S_RESET = 4'b0001; // no input seen yet
    localparam [3:0] S_A     = 4'b0010; // A has been seen (B not yet)
    localparam [3:0] S_AB    = 4'b0100; // both A and B have been seen

    // state register
    reg [3:0] state_reg;

    // --------------------------------------------------------------
    //  Sequential logic – computes next state and registered output
    // --------------------------------------------------------------
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            // asynchronous reset
            state_reg <= S_RESET;
            O         <= 1'b0;
            State     <= S_RESET;
        end else begin
            case (state_reg)
                // ----- RESET state -------------------------------------------------
                S_RESET: begin
                    if (A & ~B) begin          // A alone → move to A‑seen
                        state_reg <= S_A;
                        O         <= 1'b0;
                    end else if (A & B) begin   // A & B together → go straight to AB
                        state_reg <= S_AB;
                        O         <= 1'b1;
                    end else begin              // B alone or nothing → stay in reset
                        state_reg <= S_RESET;
                        O         <= 1'b0;
                    end
                end

                // ----- A‑SEEN state ------------------------------------------------
                S_A: begin
                    if (B) begin               // B arrived → both seen
                        state_reg <= S_AB;
                        O         <= 1'b1;
                    end else begin             // still waiting for B
                        state_reg <= S_A;
                        O         <= 1'b0;
                    end
                end

                // ----- AB state (absorbing) ----------------------------------------
                S_AB: begin
                    state_reg <= S_AB;
                    O         <= 1'b1;
                end

                // ----- default (should never happen) -------------------------------
                default: begin
                    state_reg <= S_RESET;
                    O         <= 1'b0;
                end
            endcase
            // expose the current one‑hot state for verification
            State <= state_reg;
        end
    end

endmodule
'''

filename = "abro_state_machine/abro_state_machine.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [84]:
!cd abro_state_machine/ && iverilog -o abro_state_machine.vvp abro_state_machine.v abro_state_machine_tb.v && vvp abro_state_machine.vvp

VCD info: dumpfile my_design.vcd opened for output.
Error: Test case 2 failed (A=1, B=0)
  Expected: O=0, State=0010
  Got: O=0, State=0001
abro_state_machine_tb.v:85: $finish called at 35000 (1ps)


In [85]:
feedback_prompt = '''
Still the same error:
Error: Test case 2 failed (A=1, B=0)
  Expected: O=0, State=0010
  Got: O=0, State=0001
Two problems: State <= state_reg makes State lag one cycle, and the design still remembers history. In my testbench the next state depends only on the current A and B, from any state:
	if (A && B) -> State 0100, O 1
	else if (A) -> State 0010, O 0
	else if (B) -> State 0001, O 0
	else -> hold State, O 0
Use State itself as the state register (no separate state_reg). Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
`timescale 1ns/1ps

module abro_state_machine (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // input A
    input  wire        B,        // input B
    output reg         O,        // registered output
    output reg [3:0]   State     // one‑hot state (for verification)
);

    // one‑hot encoding
    localparam [3:0] S_RESET = 4'b0001; // no input seen (reset state)
    localparam [3:0] S_A     = 4'b0010; // A seen, B not yet
    localparam [3:0] S_AB    = 4'b0100; // both A and B seen

    // --------------------------------------------------------------
    //  Sequential logic – state register updates each clock edge
    // --------------------------------------------------------------
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin               // asynchronous reset
            State <= S_RESET;
            O     <= 1'b0;
        end else be

In [86]:
output_verilog_code = '''
module abro_state_machine (
    input  wire        clk,      // system clock
    input  wire        rst_n,    // active‑low asynchronous reset
    input  wire        A,        // input A
    input  wire        B,        // input B
    output reg         O,        // registered output
    output reg [3:0]   State     // one‑hot state (for verification)
);

    // one‑hot encoding
    localparam [3:0] S_RESET = 4'b0001; // no input seen (reset state)
    localparam [3:0] S_A     = 4'b0010; // A seen, B not yet
    localparam [3:0] S_AB    = 4'b0100; // both A and B seen

    // --------------------------------------------------------------
    //  Sequential logic – state register updates each clock edge
    // --------------------------------------------------------------
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin               // asynchronous reset
            State <= S_RESET;
            O     <= 1'b0;
        end else begin
            // Next‑state logic depends **only** on the current A and B
            if (A & B) begin            // A && B → go to AB state, O = 1
                State <= S_AB;
                O     <= 1'b1;
            end else if (A) begin       // A alone → go to A‑seen state, O = 0
                State <= S_A;
                O     <= 1'b0;
            end else if (B) begin       // B alone → go to reset state, O = 0
                State <= S_RESET;
                O     <= 1'b0;
            end else begin              // neither A nor B → hold current state, O = 0
                State <= State;
                O     <= 1'b0;
            end
        end
    end

endmodule
'''

filename = "abro_state_machine/abro_state_machine.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [87]:
!cd abro_state_machine/ && iverilog -o abro_state_machine.vvp abro_state_machine.v abro_state_machine_tb.v && vvp abro_state_machine.vvp

VCD info: dumpfile my_design.vcd opened for output.
All test cases passed.
abro_state_machine_tb.v:90: $finish called at 165000 (1ps)


### Example 5: Dice Roller

In [88]:
!mkdir -p dice_roller
!cd dice_roller && curl -O https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/dice_roller_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3416  100  3416    0     0  19154      0 --:--:-- --:--:-- --:--:-- 19191


In [89]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a simulated dice roller. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - Die select (2-bits)
        - Roll
    - Outputs:
        - Rolled number (up to 8-bits)

The design should simulate rolling either a 4-sided, 6-sided, 8-sided, or 20-sided die, based on the input die select. It should roll when the roll input goes high and output the random number based on the number of sides of the selected die.

How would I write a design that meets these specifications?
'''

In [90]:
from openai import OpenAI

client = OpenAI(
  api_key = ""
)

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=[{"role":"user","content":verilog_generation_prompt}],
  max_tokens=4096, # limits the maximum number of tokens (words or pieces of words) that the model will generate in its response.
  stream=False
)

print(completion.choices[0].message.content)

Below is a **complete, synthesizable Verilog model** that satisfies the specification you gave.  
The design uses a small linear‑feedback shift register (LFSR) as a pseudo‑random number generator, detects a rising‑edge on the **roll** signal, maps the raw random value to the requested die size (4, 6, 8 or 20) and presents the result on an 8‑bit output.

```verilog
`timescale 1ns/1ps

module dice_roller (
    input        clk,          // system clock
    input        reset_n,      // active‑low asynchronous reset
    input [1:0]  die_sel,      // 2‑bit die selector
    input        roll,         // pulse that initiates a roll
    output reg [7:0] rolled_number   // result (1‑N, N = sides of die)
);

    // -----------------------------------------------------------------
    // 1.  Edge detector for the roll signal (rising‑edge only)
    // -----------------------------------------------------------------
    reg roll_d;                     // delayed version of roll
    wire roll_edge

In [93]:
output_verilog_code = '''
module dice_roller (
    input        clk,          // system clock
    input        reset_n,      // active‑low asynchronous reset
    input [1:0]  die_sel,      // 2‑bit die selector
    input        roll,         // pulse that initiates a roll
    output reg [7:0] rolled_number   // result (1‑N, N = sides of die)
);

    // -----------------------------------------------------------------
    // 1.  Edge detector for the roll signal (rising‑edge only)
    // -----------------------------------------------------------------
    reg roll_d;                     // delayed version of roll
    wire roll_edge = roll & ~roll_d; // high for one clock when roll goes 0→1

    // -----------------------------------------------------------------
    // 2.  Simple 16‑bit LFSR (maximal length polynomial x^16+x^14+x^13+x^11+1)
    // -----------------------------------------------------------------
    reg [15:0] lfsr;                // LFSR state
    // Seed with a non‑zero value on reset (any non‑zero works)
    localparam LFSR_SEED = 16'hACE1;

    // LFSR next‑state logic
    wire lfsr_fb = lfsr[15] ^ lfsr[13] ^ lfsr[12] ^ lfsr[10];
    wire [15:0] lfsr_next = {lfsr[14:0], lfsr_fb};

    // -----------------------------------------------------------------
    // 3.  Main sequential block – reset, LFSR update, roll handling
    // -----------------------------------------------------------------
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin            // ----- asynchronous reset -----
            lfsr     <= LFSR_SEED;
            roll_d   <= 1'b0;
            rolled_number <= 8'h0;
        end else begin                 // ----- normal operation -----
            // 1) capture roll for edge detection next cycle
            roll_d   <= roll;

            // 2) advance the LFSR every clock (free‑running)
            lfsr     <= lfsr_next;

            // 3) on a rising edge of roll, produce a die result
            if (roll_edge) begin
                // ----- decide how many sides the selected die has -----
                integer sides;
                case (die_sel)
                    2'b00: sides = 4;   // d4
                    2'b01: sides = 6;   // d6
                    2'b10: sides = 8;   // d8
                    2'b11: sides = 20;  // d20
                    default: sides = 4; // fallback (should never happen)
                endcase

                // ----- obtain a random number in the range 0‑31 -----
                // (5 bits are enough to cover the largest die, 20)
                reg [4:0] rand5;
                rand5 = lfsr[4:0];      // take the low 5 bits of the LFSR

                // ----- reduce the 0‑31 value to 0‑(sides‑1) -----
                // A small, synthesizable reduction loop.
                // Because the divisor is at most 20, the loop executes
                // at most once (31‑20 = 11) – still combinational.
                integer tmp;
                tmp = rand5;
                while (tmp >= sides) begin
                    tmp = tmp - sides;
                end

                // ----- final result is 1‑sized die -----
                rolled_number = tmp + 8'd1;   // convert 0‑(N‑1) → 1‑N
            end
        end
    end

endmodule
'''
filename = "dice_roller/dice_roller.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [94]:
!cd dice_roller/ && iverilog -o dice_roller.vvp dice_roller.v dice_roller_tb.v && vvp dice_roller.vvp

dice_roller.v:1: error: Variable declaration in unnamed block requires SystemVerilog.
dice_roller.v:56: syntax error
dice_roller.v:56: error: Malformed statement
dice_roller.v:63: syntax error
dice_roller.v:63: error: Malformed statement


In [95]:
feedback_prompt = '''
Your design does not compile with iverilog:
dice_roller.v:58: syntax error
dice_roller.v:58: error: Malformed statement
dice_roller.v:65: syntax error
dice_roller.v:65: error: Malformed statement
Variables are being declared in the middle of the always block; please declare them at module level. Also, my testbench uses the ports clk, rst_n, die_select [1:0], roll, rolled_number [7:0]. Please rename the ports to match. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
`timescale 1ns/1ps

module dice_roller (
    input        clk,          // system clock
    input        rst_n,        // active‑low asynchronous reset
    input [1:0]  die_select,   // die selector
    input        roll,         // pulse that initiates a roll
    output reg [7:0] rolled_number   // result (1‑N)
);

    // -------------------------------------------------------------
    // 1.  Edge detector for the roll signal (rising‑edge only)
    // -------------------------------------------------------------
    reg roll_d;                     // delayed version of roll
    wire roll_edge = roll & ~roll_d; // high for one clock when roll goes 0→1

    // -------------------------------------------------------------
    // 2.  16‑bit LFSR (maximal length polynomial x^16+x^14+x^13+x^11+1)
    // -------------------------------------------------------------
    reg [15:0] lfsr;                // LFSR state
    localparam LFSR_SEED = 16'hACE1; // any non‑zero seed works

 

In [96]:
output_verilog_code = '''
module dice_roller (
    input        clk,          // system clock
    input        rst_n,        // active‑low asynchronous reset
    input [1:0]  die_select,   // die selector
    input        roll,         // pulse that initiates a roll
    output reg [7:0] rolled_number   // result (1‑N)
);

    // -------------------------------------------------------------
    // 1.  Edge detector for the roll signal (rising‑edge only)
    // -------------------------------------------------------------
    reg roll_d;                     // delayed version of roll
    wire roll_edge = roll & ~roll_d; // high for one clock when roll goes 0→1

    // -------------------------------------------------------------
    // 2.  16‑bit LFSR (maximal length polynomial x^16+x^14+x^13+x^11+1)
    // -------------------------------------------------------------
    reg [15:0] lfsr;                // LFSR state
    localparam LFSR_SEED = 16'hACE1; // any non‑zero seed works

    wire lfsr_fb = lfsr[15] ^ lfsr[13] ^ lfsr[12] ^ lfsr[10];
    wire [15:0] lfsr_next = {lfsr[14:0], lfsr_fb};

    // -------------------------------------------------------------
    // 3.  Helper registers used inside the always block
    // -------------------------------------------------------------
    integer sides;                  // number of sides of the selected die
    reg  [4:0] rand5;               // 5‑bit random value (0‑31)
    reg  [4:0] tmp;                 // temporary for modulo reduction

    // -------------------------------------------------------------
    // 4.  Main sequential logic
    // -------------------------------------------------------------
    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin                     // asynchronous reset
            lfsr     <= LFSR_SEED;
            roll_d   <= 1'b0;
            rolled_number <= 8'h0;
        end else begin                        // normal operation
            // capture roll for edge detection next cycle
            roll_d   <= roll;

            // free‑running LFSR update
            lfsr     <= lfsr_next;

            // on a rising edge of roll, produce a die result
            if (roll_edge) begin
                // ----- map die_select to number of sides -----
                case (die_select)
                    2'b00: sides = 4;   // d4
                    2'b01: sides = 6;   // d6
                    2'b10: sides = 8;   // d8
                    2'b11: sides = 20;  // d20
                    default: sides = 4; // fallback (should never happen)
                endcase

                // ----- take 5 LSBs of the LFSR as a raw random number -----
                rand5 = lfsr[4:0];      // value in 0‑31

                // ----- reduce to the range 0‑(sides‑1) -----
                tmp = rand5;
                while (tmp >= sides) begin
                    tmp = tmp - sides;
                end

                // ----- convert to 1‑based die output -----
                rolled_number = tmp + 8'd1;
            end
        end
    end

endmodule
'''
filename = "dice_roller/dice_roller.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [97]:
!cd dice_roller/ && iverilog -o dice_roller.vvp dice_roller.v dice_roller_tb.v && vvp dice_roller.vvp

VCD info: dumpfile my_design.vcd opened for output.
Results for die_select 00:
  Rolled           1:        242 times
  Rolled           2:        265 times
  Rolled           3:        246 times
  Rolled           4:        247 times
Results for die_select 01:
  Rolled           1:        199 times
  Rolled           2:        189 times
  Rolled           3:        154 times
  Rolled           4:        184 times
  Rolled           5:        134 times
  Rolled           6:        140 times
Results for die_select 10:
  Rolled           1:        117 times
  Rolled           2:        106 times
  Rolled           3:        148 times
  Rolled           4:        129 times
  Rolled           5:        103 times
  Rolled           6:        130 times
  Rolled           7:        137 times
  Rolled           8:        130 times
Results for die_select 11:
  Rolled           1:         62 times
  Rolled           2:         63 times
  Rolled           3:         64 times
  Rolled           4:

### Example 6: LFSR

In [98]:
!mkdir -p lfsr
!cd lfsr && curl -O https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/lfsr_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1809  100  1809    0     0  12174      0 --:--:-- --:--:-- --:--:-- 12222


In [99]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for an LFSR. It must meet the following specifications:
	- Inputs:
		- Clock
        - Active-low reset
	- Outputs:
		- Data (8-bits)

The initial state should be 10001010, and the taps should be at locations 1, 4, 6, and 7.

How would I write a design that meets these specifications?
'''

In [100]:
from openai import OpenAI

client = OpenAI(
  api_key = ""
)

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=[{"role":"user","content":verilog_generation_prompt}],
  max_tokens=4096, # limits the maximum number of tokens (words or pieces of words) that the model will generate in its response.
  stream=False
)

print(completion.choices[0].message.content)

Below is a compact, synthesizable Verilog implementation that satisfies all of the requirements you listed:

* **Clock** – synchronous clock input (`clk`)  
* **Active‑low reset** – `rst_n` (logic 0 forces the register to its initial value)  
* **8‑bit data output** – `data` (the current contents of the LFSR)  
* **Initial state** – `10001010` (binary)  
* **Tap positions** – bits 1, 4, 6 and 7 (LSB = bit 0)  

The LFSR is built as a **left‑shifting** register: on each clock cycle the register is shifted left by one position and the new LSB is the XOR of the tapped bits.  (If you prefer a right‑shifting LFSR, the commented alternative is also provided.)

```verilog
`timescale 1ns/1ps

module lfsr #
(
    // -----------------------------------------------------------------
    // Parameters – easy to change if you ever need a different seed or width
    // -----------------------------------------------------------------
    parameter integer DWIDTH   = 8,                     // registe

In [101]:
output_verilog_code = '''
module lfsr #
(
    // -----------------------------------------------------------------
    // Parameters – easy to change if you ever need a different seed or width
    // -----------------------------------------------------------------
    parameter integer DWIDTH   = 8,                     // register width
    parameter [DWIDTH-1:0] INIT_STATE = 8'b10001010,   // 10001010
    // Tap positions (LSB = bit 0).  Change this vector if you want a different polynomial.
    parameter [DWIDTH-1:0] TAP_MASK   = 8'b10110010    // bits 1,4,6,7 set → 0b1011_0010
)
(
    input  wire                 clk,    // system clock
    input  wire                 rst_n,  // active‑low reset (0 = reset)
    output reg  [DWIDTH-1:0]    data    // 8‑bit LFSR output
);

// -----------------------------------------------------------------
// Internal state register
// -----------------------------------------------------------------
reg [DWIDTH-1:0] state;

// -----------------------------------------------------------------
// Synchronous reset (active low) + LFSR update
// -----------------------------------------------------------------
always @(posedge clk or negedge rst_n) begin
    if (!rst_n) begin                     // reset asserted → load seed
        state <= INIT_STATE;
    end else begin
        // ---------------------------------------------------------
        // Compute the feedback bit: XOR of all tapped bits.
        // The tap mask tells us which bits participate in the XOR.
        // ---------------------------------------------------------
        wire feedback = ^(state & TAP_MASK);   // reduction XOR of masked bits

        // ---------------------------------------------------------
        // Left‑shift the register and insert the feedback into LSB.
        // (For a right‑shifting LFSR replace the line below with:
        //   state <= {feedback, state[DWIDTH-1:1]};
        // ---------------------------------------------------------
        state <= {state[DWIDTH-2:0], feedback};
    end
end

// -----------------------------------------------------------------
// Output assignment – simply expose the internal state
// -----------------------------------------------------------------
assign data = state;

endmodule
'''
filename = "lfsr/lfsr.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [102]:
!cd lfsr/ && iverilog -o lfsr.vvp lfsr.v lfsr_tb.v && vvp lfsr.vvp

lfsr.v:34: syntax error
lfsr.v:34: Syntax in assignment statement l-value.


In [103]:
feedback_prompt = '''
Your design does not compile with iverilog:
lfsr.v:35: syntax error
lfsr.v:35: Syntax in assignment statement l-value.
A wire cannot be declared inside the always block. Also, data is declared output reg but driven with assign, and my testbench uses the ports clk, reset_n, data [7:0]. Please fix these. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
module lfsr (
    input  wire        clk,      // system clock
    input  wire        reset_n,  // active‑low reset
    output wire [7:0]  data      // 8‑bit LFSR output
);

    // -------------------------------------------------------------
    // Internal state register
    // -------------------------------------------------------------
    reg [7:0] state;

    // Seed and tap mask (taps at bits 1,4,6,7 → 1011_0010b)
    parameter [7:0] INIT_STATE = 8'b10001010;
    parameter [7:0] TAP_MASK   = 8'b10110010;

    // -------------------------------------------------------------
    // Synchronous reset + LFSR update (left‑shift, feedback into LSB)
    // -------------------------------------------------------------
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            state <= INIT_STATE;                 // reset to seed
        end else begin
            // feedback = XOR of tapped bits
            state <= {state[6:0], ^(state & TAP_

In [104]:
output_verilog_code = '''
module lfsr (
    input  wire        clk,      // system clock
    input  wire        reset_n,  // active‑low reset
    output wire [7:0]  data      // 8‑bit LFSR output
);

    // -------------------------------------------------------------
    // Internal state register
    // -------------------------------------------------------------
    reg [7:0] state;

    // Seed and tap mask (taps at bits 1,4,6,7 → 1011_0010b)
    parameter [7:0] INIT_STATE = 8'b10001010;
    parameter [7:0] TAP_MASK   = 8'b10110010;

    // -------------------------------------------------------------
    // Synchronous reset + LFSR update (left‑shift, feedback into LSB)
    // -------------------------------------------------------------
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            state <= INIT_STATE;                 // reset to seed
        end else begin
            // feedback = XOR of tapped bits
            state <= {state[6:0], ^(state & TAP_MASK)};
        end
    end

    // -------------------------------------------------------------
    // Output assignment
    // -------------------------------------------------------------
    assign data = state;

endmodule
'''
filename = "lfsr/lfsr.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [105]:
!cd lfsr/ && iverilog -o lfsr.vvp lfsr.v lfsr_tb.v && vvp lfsr.vvp

Error: At time                15000, data = 00010100, expected = 00010101
lfsr_tb.v:61: $finish called at 15000 (1ps)


In [106]:
feedback_prompt = '''
Now the testbench reports: "Error: At time 15000, data = 00010100, expected = 00010101".
The tap locations 1, 4, 6, 7 are numbered starting from 1, not 0. Please fix the taps. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
module lfsr (
    input  wire        clk,      // system clock
    input  wire        reset_n,  // active‑low reset
    output wire [7:0]  data      // 8‑bit LFSR output
);

    // -------------------------------------------------------------
    // Internal state register
    // -------------------------------------------------------------
    reg [7:0] state;

    // Initial state = 10001010b
    parameter [7:0] INIT_STATE = 8'b10001010;

    // Tap locations (numbered from 1 starting at the LSB):
    //   taps at positions 1,4,6,7  -> bits 0,3,5,6
    //   mask = b0110_1001
    parameter [7:0] TAP_MASK   = 8'b01101001;

    // -------------------------------------------------------------
    // Synchronous reset + LFSR update (left‑shift, feedback into LSB)
    // -------------------------------------------------------------
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            state <= INIT_STATE;                 // reset to seed
    

In [107]:
output_verilog_code = '''
module lfsr (
    input  wire        clk,      // system clock
    input  wire        reset_n,  // active‑low reset
    output wire [7:0]  data      // 8‑bit LFSR output
);

    // -------------------------------------------------------------
    // Internal state register
    // -------------------------------------------------------------
    reg [7:0] state;

    // Initial state = 10001010b
    parameter [7:0] INIT_STATE = 8'b10001010;

    // Tap locations (numbered from 1 starting at the LSB):
    //   taps at positions 1,4,6,7  -> bits 0,3,5,6
    //   mask = b0110_1001
    parameter [7:0] TAP_MASK   = 8'b01101001;

    // -------------------------------------------------------------
    // Synchronous reset + LFSR update (left‑shift, feedback into LSB)
    // -------------------------------------------------------------
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            state <= INIT_STATE;                 // reset to seed
        end else begin
            // feedback = XOR of tapped bits
            state <= {state[6:0], ^(state & TAP_MASK)};
        end
    end

    // -------------------------------------------------------------
    // Output assignment
    // -------------------------------------------------------------
    assign data = state;

endmodule
'''
filename = "lfsr/lfsr.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [108]:
!cd lfsr/ && iverilog -o lfsr.vvp lfsr.v lfsr_tb.v && vvp lfsr.vvp

Simulation successful. All test cases passed.
lfsr_tb.v:67: $finish called at 2565000 (1ps)


### Example 7: Sequence Generator

In [109]:
!mkdir -p sequence_generator
!cd sequence_generator && curl -O https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_generator_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2344  100  2344    0     0  11916      0 --:--:-- --:--:-- --:--:-- 11959


In [110]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a sequence generator. It must meet the following specifications:
	- Inputs:
		- Clock
		- Active-low reset
		- Enable
	- Outputs:
		- Data (8 bits)

While enabled, it should generate an output sequence of the following hexadecimal values and then repeat:
	- 0xAF
	- 0xBC
	- 0xE2
	- 0x78
	- 0xFF
	- 0xE2
	- 0x0B
	- 0x8D

How would I write a design that meets these specifications?
'''

In [111]:
from openai import OpenAI

client = OpenAI(
  api_key = ""
)

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=[{"role":"user","content":verilog_generation_prompt}],
  max_tokens=4096, # limits the maximum number of tokens (words or pieces of words) that the model will generate in its response.
  stream=False
)

print(completion.choices[0].message.content)

**Verilog design – 8‑bit sequence generator**

```verilog
//--------------------------------------------------------------
//  Sequence Generator
//  - Clock   : clk
//  - Reset   : rst_n   (active‑low)
//  - Enable  : en
//  - Output  : data[7:0]   (8‑bit sequence)
//
//  Sequence (hex): AF BC E2 78 FF E2 0B 8D  (repeats)
//--------------------------------------------------------------
module seq_gen (
    input  wire       clk,      // system clock
    input  wire       rst_n,    // active‑low asynchronous reset
    input  wire       en,       // enable (high = run)
    output reg  [7:0] data      // generated byte
);

// -----------------------------------------------------------------
// 1) Store the constant sequence in a packed array.
//    Using a `localparam` lets the synthesizer see the values as
    // constants and infer a small ROM if desired.
// -----------------------------------------------------------------
localparam [7:0] SEQ [0:7] = {
    8'hAF,
    8'hBC,
    8'hE2,

In [112]:
output_verilog_code = '''
module seq_gen (
    input  wire       clk,      // system clock
    input  wire       rst_n,    // active‑low asynchronous reset
    input  wire       en,       // enable (high = run)
    output reg  [7:0] data      // generated byte
);

// -----------------------------------------------------------------
// 1) Store the constant sequence in a packed array.
//    Using a `localparam` lets the synthesizer see the values as
    // constants and infer a small ROM if desired.
// -----------------------------------------------------------------
localparam [7:0] SEQ [0:7] = {
    8'hAF,
    8'hBC,
    8'hE2,
    8'h78,
    8'hFF,
    8'hE2,
    8'h0B,
    8'h8D
};

// -----------------------------------------------------------------
// 2) Pointer that indexes into SEQ.
//    It is incremented on each enabled clock tick and wraps
//    back to zero after the last element.
// -----------------------------------------------------------------
integer ptr;   // synthesis‑safe: will be inferred as a small counter

// -----------------------------------------------------------------
// 3) Sequential logic (reset + enable)
// -----------------------------------------------------------------
always @(posedge clk or negedge rst_n) begin
    if (!rst_n) begin               // active‑low reset
        data <= 8'h0;               // optional: clear output on reset
        ptr  <= 0;                  // restart sequence from the first entry
    end
    else if (en) begin              // only advance when enabled
        data <= SEQ[ptr];           // output the current value
        ptr  <= (ptr + 1) % 8;      // wrap‑around counter (0…7)
    end
    // If en == 0 we hold the current value (no change)
end

endmodule
'''
filename = "sequence_generator/sequence_generator.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [113]:
!cd sequence_generator/ && iverilog -o sequence_generator.vvp sequence_generator.v sequence_generator_tb.v && vvp sequence_generator.vvp

sequence_generator.v:14: error: localparam must have a value.
sequence_generator.v:14: syntax error
sequence_generator.v:23: error: Invalid module item.


In [114]:
feedback_prompt = '''
Your design does not compile with iverilog:
sequence_generator.v:7: error: localparam must have a value.
sequence_generator.v:7: syntax error
sequence_generator.v:9: error: Invalid module item.
iverilog does not support localparam arrays; please avoid arrays. Also, my testbench instantiates the module as sequence_generator with ports clk, reset_n, enable, data [7:0]. Please use these names. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
module sequence_generator (
    input  wire       clk,
    input  wire       reset_n,   // active‑low asynchronous reset
    input  wire       enable,
    output reg  [7:0] data
);

// ---------------------------------------------------------------
//  ROM holding the constant 8‑byte sequence
// ---------------------------------------------------------------
reg [7:0] seq [0:7];
integer   ptr;                     // will be inferred as a 3‑bit counter

// Initialise the ROM (supported by Icarus Verilog)
initial begin
    seq[0] = 8'hAF;
    seq[1] = 8'hBC;
    seq[2] = 8'hE2;
    seq[3] = 8'h78;
    seq[4] = 8'hFF;
    seq[5] = 8'hE2;
    seq[6] = 8'h0B;
    seq[7] = 8'h8D;
end

// ---------------------------------------------------------------
//  Sequential logic: reset + enable‑controlled pointer advance
// ---------------------------------------------------------------
always @(posedge clk or negedge reset_n) begin
    if (!reset_n) begin               // active‑low rese

In [116]:
output_verilog_code = '''
module sequence_generator (
    input  wire       clk,
    input  wire       reset_n,   // active‑low asynchronous reset
    input  wire       enable,
    output reg  [7:0] data
);

// ---------------------------------------------------------------
//  ROM holding the constant 8‑byte sequence
// ---------------------------------------------------------------
reg [7:0] seq [0:7];
integer   ptr;                     // will be inferred as a 3‑bit counter

// Initialise the ROM (supported by Icarus Verilog)
initial begin
    seq[0] = 8'hAF;
    seq[1] = 8'hBC;
    seq[2] = 8'hE2;
    seq[3] = 8'h78;
    seq[4] = 8'hFF;
    seq[5] = 8'hE2;
    seq[6] = 8'h0B;
    seq[7] = 8'h8D;
end

// ---------------------------------------------------------------
//  Sequential logic: reset + enable‑controlled pointer advance
// ---------------------------------------------------------------
always @(posedge clk or negedge reset_n) begin
    if (!reset_n) begin               // active‑low reset
        data <= 8'h0;                 // optional: clear output on reset
        ptr  <= 0;                    // restart sequence from first entry
    end else if (enable) begin        // advance only when enabled
        data <= seq[ptr];             // output current value
        ptr  <= (ptr + 1) % 8;        // wrap‑around counter (0‑7)
    end
    // If enable == 0 we hold the current output (no change)
end

endmodule
'''
filename = "sequence_generator/sequence_generator.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [117]:
!cd sequence_generator/ && iverilog -o sequence_generator.vvp sequence_generator.v sequence_generator_tb.v && vvp sequence_generator.vvp

Error: Mismatch at position 0. Expected: af, Got: 00
sequence_generator_tb.v:58: $finish called at 35000 (1ps)


In [118]:
feedback_prompt = '''
Now the testbench reports: "Error: Mismatch at position 0. Expected: af, Got: 00".
The testbench expects 0xAF on data right after reset, before any enabled clock edge. Please fix the design. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
module sequence_generator (
    input  wire       clk,
    input  wire       reset_n,   // active‑low asynchronous reset
    input  wire       enable,
    output reg  [7:0] data
);

    // ---------------------------------------------------------------
    // ROM holding the constant 8‑byte sequence
    // ---------------------------------------------------------------
    reg [7:0] seq [0:7];
    integer   ptr;                     // 3‑bit counter (0‑7)

    initial begin
        seq[0] = 8'hAF;
        seq[1] = 8'hBC;
        seq[2] = 8'hE2;
        seq[3] = 8'h78;
        seq[4] = 8'hFF;
        seq[5] = 8'hE2;
        seq[6] = 8'h0B;
        seq[7] = 8'h8D;
    end

    // ---------------------------------------------------------------
    // Sequential logic: reset + enable‑controlled pointer advance
    // ---------------------------------------------------------------
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin                 // asy

In [119]:
output_verilog_code = '''
module sequence_generator (
    input  wire       clk,
    input  wire       reset_n,   // active‑low asynchronous reset
    input  wire       enable,
    output reg  [7:0] data
);

    // ---------------------------------------------------------------
    // ROM holding the constant 8‑byte sequence
    // ---------------------------------------------------------------
    reg [7:0] seq [0:7];
    integer   ptr;                     // 3‑bit counter (0‑7)

    initial begin
        seq[0] = 8'hAF;
        seq[1] = 8'hBC;
        seq[2] = 8'hE2;
        seq[3] = 8'h78;
        seq[4] = 8'hFF;
        seq[5] = 8'hE2;
        seq[6] = 8'h0B;
        seq[7] = 8'h8D;
    end

    // ---------------------------------------------------------------
    // Sequential logic: reset + enable‑controlled pointer advance
    // ---------------------------------------------------------------
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin                 // asynchronous active‑low reset
            data <= seq[0];                 // output first value immediately
            ptr  <= 0;                      // restart sequence
        end else if (enable) begin          // advance only when enabled
            data <= seq[ptr];               // output current value
            ptr  <= (ptr + 1) % 8;          // wrap‑around counter
        end
        // If enable == 0 we hold the current output (no change)
    end

endmodule
'''
filename = "sequence_generator/sequence_generator.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [120]:
!cd sequence_generator/ && iverilog -o sequence_generator.vvp sequence_generator.v sequence_generator_tb.v && vvp sequence_generator.vvp

Error: Mismatch at position 1. Expected: bc, Got: af
sequence_generator_tb.v:58: $finish called at 45000 (1ps)


In [121]:
feedback_prompt = '''
Now the testbench reports: "Error: Mismatch at position 1. Expected: bc, Got: af".
0xAF is output twice: once at reset and again on the first enabled clock edge. Each enabled clock edge should move to the next value. Please fix the design. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
module sequence_generator (
    input  wire       clk,
    input  wire       reset_n,   // active‑low asynchronous reset
    input  wire       enable,
    output reg  [7:0] data
);

    // ROM with the constant sequence
    reg [7:0] seq [0:7];
    integer   ptr;                 // 3‑bit counter (0‑7)

    initial begin
        seq[0] = 8'hAF;
        seq[1] = 8'hBC;
        seq[2] = 8'hE2;
        seq[3] = 8'h78;
        seq[4] = 8'hFF;
        seq[5] = 8'hE2;
        seq[6] = 8'h0B;
        seq[7] = 8'h8D;
    end

    // Sequential logic: reset clears the pointer; on each enabled clock edge
    // we output the current value and advance the pointer.
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin               // asynchronous active‑low reset
            ptr  <= 0;                    // restart sequence
            data <= 8'b0;                 // no meaningful output while reset is asserted
        end else if (enable) begin        // advan

In [122]:
output_verilog_code = '''
module sequence_generator (
    input  wire       clk,
    input  wire       reset_n,   // active‑low asynchronous reset
    input  wire       enable,
    output reg  [7:0] data
);

    // ROM with the constant sequence
    reg [7:0] seq [0:7];
    integer   ptr;                 // 3‑bit counter (0‑7)

    initial begin
        seq[0] = 8'hAF;
        seq[1] = 8'hBC;
        seq[2] = 8'hE2;
        seq[3] = 8'h78;
        seq[4] = 8'hFF;
        seq[5] = 8'hE2;
        seq[6] = 8'h0B;
        seq[7] = 8'h8D;
    end

    // Sequential logic: reset clears the pointer; on each enabled clock edge
    // we output the current value and advance the pointer.
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin               // asynchronous active‑low reset
            ptr  <= 0;                    // restart sequence
            data <= 8'b0;                 // no meaningful output while reset is asserted
        end else if (enable) begin        // advance only when enabled
            data <= seq[ptr];             // output current sequence element
            ptr  <= (ptr + 1) % 8;        // wrap‑around counter
        end
        // If enable == 0 we hold the previous output (no change)
    end

endmodule
'''
filename = "sequence_generator/sequence_generator.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [123]:
!cd sequence_generator/ && iverilog -o sequence_generator.vvp sequence_generator.v sequence_generator_tb.v && vvp sequence_generator.vvp

Error: Mismatch at position 0. Expected: af, Got: 00
sequence_generator_tb.v:58: $finish called at 35000 (1ps)


In [124]:
feedback_prompt = '''
This is the same as your first version and fails again with "Mismatch at position 0. Expected: af, Got: 00".
Both conditions must hold: 0xAF on data right after reset, and 0xBC after the first enabled clock edge. Make data combinational from the pointer (data = seq[ptr], with ptr reset to 0 and incremented on each enabled clock edge) instead of registering data. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
module sequence_generator (
    input  wire       clk,
    input  wire       reset_n,   // active‑low asynchronous reset
    input  wire       enable,
    output reg  [7:0] data
);

    // ROM with the constant sequence
    reg [7:0] seq [0:7];
    reg [2:0] ptr;                 // 3‑bit pointer (0‑7)

    initial begin
        seq[0] = 8'hAF;
        seq[1] = 8'hBC;
        seq[2] = 8'hE2;
        seq[3] = 8'h78;
        seq[4] = 8'hFF;
        seq[5] = 8'hE2;
        seq[6] = 8'h0B;
        seq[7] = 8'h8D;
    end

    // Pointer update: reset clears it; on each enabled clock edge it advances.
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            ptr <= 3'b0;                 // reset to first element
        end else if (enable) begin
            ptr <= (ptr + 1) % 8;        // wrap‑around counter
        end
        // If enable == 0 the pointer holds its value.
    end

    // Combinational output directly from the ROM.
    assign dat

In [125]:
output_verilog_code = '''
module sequence_generator (
    input  wire       clk,
    input  wire       reset_n,   // active‑low asynchronous reset
    input  wire       enable,
    output reg  [7:0] data
);

    // ROM with the constant sequence
    reg [7:0] seq [0:7];
    reg [2:0] ptr;                 // 3‑bit pointer (0‑7)

    initial begin
        seq[0] = 8'hAF;
        seq[1] = 8'hBC;
        seq[2] = 8'hE2;
        seq[3] = 8'h78;
        seq[4] = 8'hFF;
        seq[5] = 8'hE2;
        seq[6] = 8'h0B;
        seq[7] = 8'h8D;
    end

    // Pointer update: reset clears it; on each enabled clock edge it advances.
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            ptr <= 3'b0;                 // reset to first element
        end else if (enable) begin
            ptr <= (ptr + 1) % 8;        // wrap‑around counter
        end
        // If enable == 0 the pointer holds its value.
    end

    // Combinational output directly from the ROM.
    assign data = seq[ptr];

endmodule
'''
filename = "sequence_generator/sequence_generator.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [126]:
!cd sequence_generator/ && iverilog -o sequence_generator.vvp sequence_generator.v sequence_generator_tb.v && vvp sequence_generator.vvp

sequence_generator.v:35: error: reg data; cannot be driven by primitives or continuous assignment.
1 error(s) during elaboration.


In [127]:
feedback_prompt = '''
Now iverilog reports:
sequence_generator.v:17: error: reg data; cannot be driven by primitives or continuous assignment.
Please fix this. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
module sequence_generator (
    input  wire       clk,
    input  wire       reset_n,   // active‑low asynchronous reset
    input  wire       enable,
    output wire [7:0] data
);

    // ROM with the constant sequence
    reg [7:0] seq [0:7];
    reg [2:0] ptr;                 // 3‑bit pointer (0‑7)

    initial begin
        seq[0] = 8'hAF;
        seq[1] = 8'hBC;
        seq[2] = 8'hE2;
        seq[3] = 8'h78;
        seq[4] = 8'hFF;
        seq[5] = 8'hE2;
        seq[6] = 8'h0B;
        seq[7] = 8'h8D;
    end

    // Pointer update: reset clears it; on each enabled clock edge it advances.
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            ptr <= 3'b0;                 // reset to first element
        end else if (enable) begin
            ptr <= (ptr + 1) % 8;        // wrap‑around counter
        end
        // If enable == 0 the pointer holds its value.
    end

    // Combinational output directly from the ROM.
    assign dat

In [130]:
output_verilog_code = '''
module sequence_generator (
    input  wire       clk,
    input  wire       reset_n,   // active‑low asynchronous reset
    input  wire       enable,
    output wire [7:0] data
);

    // ROM with the constant sequence
    reg [7:0] seq [0:7];
    reg [2:0] ptr;                 // 3‑bit pointer (0‑7)

    initial begin
        seq[0] = 8'hAF;
        seq[1] = 8'hBC;
        seq[2] = 8'hE2;
        seq[3] = 8'h78;
        seq[4] = 8'hFF;
        seq[5] = 8'hE2;
        seq[6] = 8'h0B;
        seq[7] = 8'h8D;
    end

    // Pointer update: reset clears it; on each enabled clock edge it advances.
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            ptr <= 3'b0;                 // reset to first element
        end else if (enable) begin
            ptr <= (ptr + 1) % 8;        // wrap‑around counter
        end
        // If enable == 0 the pointer holds its value.
    end

    // Combinational output directly from the ROM.
    assign data = seq[ptr];

endmodule
'''
filename = "sequence_generator/sequence_generator.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [131]:
!cd sequence_generator/ && iverilog -o sequence_generator.vvp sequence_generator.v sequence_generator_tb.v && vvp sequence_generator.vvp

Test case passed: sequence generation
Test case passed: sequence repetition
Test case passed: disable sequence generator
All test cases passed
sequence_generator_tb.v:83: $finish called at 195000 (1ps)


### Example 8: Traffic Light

In [132]:
!mkdir -p traffic_light/
!cd traffic_light/ && curl -O https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/traffic_light_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2018  100  2018    0     0  12046      0 --:--:-- --:--:-- --:--:-- 12083


In [133]:
verilog_generation_prompt = '''
I am trying to create a Verilog model for a traffic light state machine. It must meet the following specifications:
    - Inputs:
        - Clock
        - Active-low reset
        - Enable
    - Outputs:
        - Red
        - Yellow
        - Green

The state machine should reset to a red light, change from red to green after 32 clock cycles, change from green to yellow after 20 clock cycles, and then change from yellow to red after 7 clock cycles.

How would I write a design that meets these specifications?
'''

In [134]:
from openai import OpenAI

client = OpenAI(
  api_key = ""
)

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=[{"role":"user","content":verilog_generation_prompt}],
  max_tokens=4096, # limits the maximum number of tokens (words or pieces of words) that the model will generate in its response.
  stream=False
)

print(completion.choices[0].message.content)

Below is a complete, synthesizable Verilog‑2001 model of the traffic‑light FSM that follows the specification you gave.

---

## 1. Design Overview  

| Signal | Direction | Description |
|--------|-----------|-------------|
| `clk`      | input  | System clock (positive edge triggered) |
| `reset_n`  | input  | **Active‑low** asynchronous reset – forces the light to RED |
| `enable`   | input  | When high the timer runs; when low the FSM holds its current state |
| `red`, `yellow`, `green` | output | One‑hot indicators for the three lights (active‑high) |

### States  

```
parameter RED   = 2'b00;
parameter GREEN = 2'b01;
parameter YELLOW= 2'b10;
```

### Timing constants  

```
parameter RED_TIME   = 32;   // stay RED   for 32 clock cycles
parameter GREEN_TIME = 20;   // stay GREEN for 20 clock cycles
parameter YELLOW_TIME= 7;    // stay YELLOW for 7 clock cycles
```

A single free‑running counter (`cnt`) is used.  
When the counter reaches the time‑out value for the **current** sta

In [135]:
output_verilog_code = '''
module traffic_light_fsm (
    input  wire        clk,      // system clock
    input  wire        reset_n,  // active‑low asynchronous reset
    input  wire        enable,   // enable counting
    output reg         red,      // RED   light (active‑high)
    output reg         yellow,   // YELLOW light (active‑high)
    output reg         green     // GREEN light (active‑high)
);

    // -----------------------------------------------------------------
    // 1) Parameter definitions (state encoding & timing)
    // -----------------------------------------------------------------
    parameter [1:0] RED   = 2'b00;
    parameter [1:0] GREEN = 2'b01;
    parameter [1:0] YELLOW= 2'b10;

    parameter integer RED_TIME   = 32;
    parameter integer GREEN_TIME = 20;
    parameter integer YELLOW_TIME= 7;

    // -----------------------------------------------------------------
    // 2) State & counter registers
    // -----------------------------------------------------------------
    reg [1:0] state, next_state;
    reg [$clog2(RED_TIME)-1:0] cnt;          // width big enough for the largest timeout
    wire [$clog2(RED_TIME)-1:0] timeout;     // timeout value for the *current* state

    // -----------------------------------------------------------------
    // 3) Combinatorial logic: next‑state & output decoding
    // -----------------------------------------------------------------
    // Determine timeout based on present state
    always @* begin
        case (state)
            RED   : timeout = RED_TIME   - 1'b1;
            GREEN : timeout = GREEN_TIME - 1'b1;
            YELLOW: timeout = YELLOW_TIME- 1'b1;
            default: timeout = 0;
        endcase
    end

    // Next‑state logic (only changes when enable is high and timeout reached)
    always @* begin
        // Default: stay in same state
        next_state = state;

        if (enable) begin
            // When the counter has reached the timeout, advance
            if (cnt == timeout) begin
                case (state)
                    RED   : next_state = GREEN;
                    GREEN : next_state = YELLOW;
                    YELLOW: next_state = RED;
                    default: next_state = RED; // safety fallback
                endcase
            end
        end
    end

    // Output decoding (one‑hot)
    always @* begin
        red    = (state == RED);
        yellow = (state == YELLOW);
        green  = (state == GREEN);
    end

    // -----------------------------------------------------------------
    // 4) Sequential logic: state update & counter
    // -----------------------------------------------------------------
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin          // async active‑low reset
            state  <= RED;
            cnt    <= '0;
        end else begin
            state  <= next_state;

            // Counter only runs when enable is high
            if (enable) begin
                // If we have reached the timeout, wrap to zero;
                // otherwise increment.
                if (cnt == timeout)
                    cnt <= '0;
                else
                    cnt <= cnt + 1'b1;
            end else begin
                // Hold the counter when enable is low
                cnt <= cnt;
            end
        end
    end

endmodule
'''
filename = "traffic_light/traffic_light.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [138]:
!cd traffic_light/ && iverilog -g2012 -o traffic_light.vvp traffic_light.v traffic_light_tb.v && vvp traffic_light.vvp

traffic_light.v:35: error: timeout is not a valid l-value in tb_traffic_light_fsm.traffic_light.
traffic_light.v:27:      : timeout is declared here as wire.
traffic_light.v:36: error: timeout is not a valid l-value in tb_traffic_light_fsm.traffic_light.
traffic_light.v:27:      : timeout is declared here as wire.
traffic_light.v:37: error: timeout is not a valid l-value in tb_traffic_light_fsm.traffic_light.
traffic_light.v:27:      : timeout is declared here as wire.
traffic_light.v:38: error: timeout is not a valid l-value in tb_traffic_light_fsm.traffic_light.
traffic_light.v:27:      : timeout is declared here as wire.
4 error(s) during elaboration.


In [139]:
feedback_prompt = '''
Your design does not compile with iverilog:
traffic_light.v:35: error: timeout is not a valid l-value in tb_traffic_light_fsm.traffic_light.
traffic_light.v:27:      : timeout is declared here as wire.
(same error for lines 36, 37, 38)
Please fix this. Only give the corrected Verilog module.
'''

# ChipChat: continue the conversation with human feedback
messages = [{"role":"user","content":verilog_generation_prompt},
            {"role":"assistant","content":completion.choices[0].message.content},
            {"role":"user","content":feedback_prompt}]

completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=messages,
  max_tokens=4096,
  stream=False
)

print(completion.choices[0].message.content)

```verilog
`timescale 1ns/1ps

module traffic_light_fsm (
    input  wire        clk,      // system clock
    input  wire        reset_n,  // active‑low asynchronous reset
    input  wire        enable,   // enable counting
    output reg         red,      // RED   light (active‑high)
    output reg         yellow,   // YELLOW light (active‑high)
    output reg         green     // GREEN light (active‑high)
);

    // -----------------------------------------------------------------
    // 1) Parameter definitions (state encoding & timing)
    // -----------------------------------------------------------------
    parameter [1:0] RED   = 2'b00;
    parameter [1:0] GREEN = 2'b01;
    parameter [1:0] YELLOW= 2'b10;

    parameter integer RED_TIME   = 32;
    parameter integer GREEN_TIME = 20;
    parameter integer YELLOW_TIME= 7;

    // -----------------------------------------------------------------
    // 2) State & counter registers
    // -----------------------------------------

In [140]:
output_verilog_code = '''
module traffic_light_fsm (
    input  wire        clk,      // system clock
    input  wire        reset_n,  // active‑low asynchronous reset
    input  wire        enable,   // enable counting
    output reg         red,      // RED   light (active‑high)
    output reg         yellow,   // YELLOW light (active‑high)
    output reg         green     // GREEN light (active‑high)
);

    // -----------------------------------------------------------------
    // 1) Parameter definitions (state encoding & timing)
    // -----------------------------------------------------------------
    parameter [1:0] RED   = 2'b00;
    parameter [1:0] GREEN = 2'b01;
    parameter [1:0] YELLOW= 2'b10;

    parameter integer RED_TIME   = 32;
    parameter integer GREEN_TIME = 20;
    parameter integer YELLOW_TIME= 7;

    // -----------------------------------------------------------------
    // 2) State & counter registers
    // -----------------------------------------------------------------
    reg [1:0] state, next_state;
    reg [$clog2(RED_TIME)-1:0] cnt;          // width big enough for the largest timeout

    // -----------------------------------------------------------------
    // 3) Function to obtain timeout for the current state
    // -----------------------------------------------------------------
    function automatic [$clog2(RED_TIME)-1:0] get_timeout;
        input [1:0] st;
        case (st)
            RED   : get_timeout = RED_TIME   - 1'b1;
            GREEN : get_timeout = GREEN_TIME - 1'b1;
            YELLOW: get_timeout = YELLOW_TIME- 1'b1;
            default: get_timeout = 0;
        endcase
    endfunction

    // Continuous assignment: timeout is a read‑only wire
    wire [$clog2(RED_TIME)-1:0] timeout = get_timeout(state);

    // -----------------------------------------------------------------
    // 4) Combinatorial logic: next‑state & output decoding
    // -----------------------------------------------------------------
    // Next‑state logic (only changes when enable is high and timeout reached)
    always @* begin
        // Default: stay in same state
        next_state = state;

        if (enable) begin
            if (cnt == timeout) begin
                case (state)
                    RED   : next_state = GREEN;
                    GREEN : next_state = YELLOW;
                    YELLOW: next_state = RED;
                    default: next_state = RED; // safety fallback
                endcase
            end
        end
    end

    // Output decoding (one‑hot)
    always @* begin
        red    = (state == RED);
        yellow = (state == YELLOW);
        green  = (state == GREEN);
    end

    // -----------------------------------------------------------------
    // 5) Sequential logic: state update & counter
    // -----------------------------------------------------------------
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin          // async active‑low reset
            state  <= RED;
            cnt    <= '0;
        end else begin
            state  <= next_state;

            // Counter only runs when enable is high
            if (enable) begin
                if (cnt == timeout)
                    cnt <= '0;
                else
                    cnt <= cnt + 1'b1;
            end else begin
                // Hold the counter when enable is low
                cnt <= cnt;
            end
        end
    end

endmodule
'''
filename = "traffic_light/traffic_light.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [141]:
!cd traffic_light/ && iverilog -g2012 -o traffic_light.vvp traffic_light.v traffic_light_tb.v && vvp traffic_light.vvp

Test case 1 passed - FSM transitioned to    RED after 0 clock cycles.
Test case 2 passed - FSM transitioned to  GREEN after 32 clock cycles.
Test case 3 passed - FSM transitioned to YELLOW after 20 clock cycles.
Test case 4 passed - FSM transitioned to    RED after 7 clock cycles.
All test cases passed!
traffic_light_tb.v:79: $finish called at 610 (1ps)


### Example 9: CPU (8-bit accumulator processor)

In [142]:
!mkdir -p cpu

In [146]:
verilog_generation_prompt = '''
Let us make a brand new microprocessor design together. We're severely constrained on space and I/O. We have to fit in 1000 standard cells of an ASIC, so I think we will need to restrict ourselves to an accumulator based 8-bit architecture with no multi-byte instructions. Given this, how do you think we should begin?
'''

In [151]:
cpu_messages = [{"role":"user","content":verilog_generation_prompt}]
completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=cpu_messages,
  max_tokens=8192,
  extra_body={"chat_template_kwargs": {"enable_thinking": False}}, # checked print(completion.usage), everything went to reasoning nothing to the answer.
  stream=False
)
cpu_messages.append({"role":"assistant","content":completion.choices[0].message.content})
print(completion.choices[0].message.content)

Great idea — designing a minimal 8-bit accumulator-based microprocessor within a strict 1000 standard cell budget is a fascinating challenge, and it's absolutely feasible with careful architectural trade-offs. Since we're constrained by both **logic density** (standard cells) and **I/O pins**, we must prioritize simplicity, regularity, and reuse.

Let’s begin by **defining our goals, constraints, and then stepping through a structured design process**.

---

### ✅ **Design Constraints Recap**
- **8-bit data width**
- **Accumulator-based architecture** (single primary register: ACC)
- **No multi-byte instructions** → all instructions fit in 8 bits (1 byte)
- **Target: ~1000 standard cells** (ASIC gate equivalent; assume ~4–6 transistors per cell → ~4000–6000 transistors)
- **Minimal I/O** → likely only a few pins: clock, reset, maybe 1–2 for serial I/O or memory enable
- **On-chip memory?** Probably not — we’ll assume external ROM/RAM via address/data buses, but we must minimize bus wid

In [149]:
print(completion.usage)

CompletionUsage(completion_tokens=16384, prompt_tokens=87, total_tokens=16471, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=16384, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=0))


Feedback round 1: choose the architecture and ask for the Verilog.

In [152]:
feedback_prompt = '''
Lets simplify it to make sure we fit in 1000 cells, keep your [7:5] opcode field and your 2-cycle FETCH/EXECUTE control:
	- Drop the stack, SWAP, immediates and the N/C flags. Use one unified program/data memory instead of separate ROM and RAM, so a jump can just use the address bits in the instruction.
	- Instruction format: [7:5] opcode, lower bits = memory address.
	- Opcodes: 000 LDA, 001 STA, 010 ADD, 011 SUB, 100 AND, 101 JMP, 110 JZ (jump if accumulator is 0), 111 HLT.
	- Start with a 32-byte memory and a 5-bit address and PC.
Please write the Verilog: module name cpu, ports clk, rst_n (active-low, asynchronous), acc_out [7:0] (the accumulator), halted. The memory must be an internal array named mem (reg [7:0] mem [0:31]) so my testbench can load programs into it. Only give the Verilog module.
'''

cpu_messages.append({"role":"user","content":feedback_prompt})
completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=cpu_messages,
  max_tokens=4096,
  stream=False
)
cpu_messages.append({"role":"assistant","content":completion.choices[0].message.content})
print(completion.choices[0].message.content)

```verilog
module cpu (
    input        clk,
    input        rst_n,      // active-low asynchronous reset
    output [7:0] acc_out,
    output       halted
);

    // 5-bit PC for 32-byte memory
    reg [4:0]  PC;
    // 8-bit accumulator
    reg [7:0]  ACC;
    // Halt flag
    reg        halted;
    // FSM state: 0 = FETCH, 1 = EXECUTE
    reg [1:0]  state;
    // Instruction register
    reg [7:0]  IR;
    // Internal memory (32x8)
    reg [7:0]  mem [0:31];

    // Output accumulator value
    assign acc_out = ACC;

    // State encoding
    localparam FETCH = 2'b00;
    localparam EXEC  = 2'b01;

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            // Synchronous reset (async due to negedge)
            PC    <= 5'h0;
            ACC   <= 8'h0;
            halted<= 1'b0;
            state <= FETCH;
            IR    <= 8'h0;
        end else begin
            if (halted) begin
                // Remain halted; do nothing
            end else beg

In [153]:
output_verilog_code = '''
module cpu (
    input        clk,
    input        rst_n,      // active-low asynchronous reset
    output [7:0] acc_out,
    output       halted
);

    // 5-bit PC for 32-byte memory
    reg [4:0]  PC;
    // 8-bit accumulator
    reg [7:0]  ACC;
    // Halt flag
    reg        halted;
    // FSM state: 0 = FETCH, 1 = EXECUTE
    reg [1:0]  state;
    // Instruction register
    reg [7:0]  IR;
    // Internal memory (32x8)
    reg [7:0]  mem [0:31];

    // Output accumulator value
    assign acc_out = ACC;

    // State encoding
    localparam FETCH = 2'b00;
    localparam EXEC  = 2'b01;

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            // Synchronous reset (async due to negedge)
            PC    <= 5'h0;
            ACC   <= 8'h0;
            halted<= 1'b0;
            state <= FETCH;
            IR    <= 8'h0;
        end else begin
            if (halted) begin
                // Remain halted; do nothing
            end else begin
                case (state)
                    FETCH: begin
                        // Fetch instruction from memory[PC]
                        IR <= mem[PC];
                        state <= EXEC;
                    end
                    EXEC: begin
                        // Execute instruction based on opcode IR[7:5]
                        case (IR[7:5])
                            3'b000: // LDA: ACC <- mem[addr]
                                ACC <= mem[IR[4:0]];
                            3'b001: // STA: mem[addr] <- ACC
                                mem[IR[4:0]] <= ACC;
                            3'b010: // ADD: ACC <- ACC + mem[addr]
                                ACC <= ACC + mem[IR[4:0]];
                            3'b011: // SUB: ACC <- ACC - mem[addr]
                                ACC <= ACC - mem[IR[4:0]];
                            3'b100: // AND: ACC <- ACC & mem[addr]
                                ACC <= ACC & mem[IR[4:0]];
                            3'b101: // JMP: PC <- addr
                                PC <= IR[4:0];
                            3'b110: // JZ: if ACC==0 then PC <- addr else PC <- PC+1
                                if (ACC == 8'h0)
                                    PC <= IR[4:0];
                                else
                                    PC <= PC + 1;
                            3'b111: // HLT: set halted flag
                                halted <= 1'b1;
                        endcase

                        // Default PC increment for non-branch, non-halt instructions
                        if (IR[7:5] == 3'b000 || IR[7:5] == 3'b001 ||
                            IR[7:5] == 3'b010 || IR[7:5] == 3'b011 ||
                            IR[7:5] == 3'b100) begin
                            PC <= PC + 1;
                        end

                        // Return to fetch state
                        state <= FETCH;
                    end
                endcase
            end
        end
    end
endmodule
'''
filename = "cpu/cpu.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

Testbench: loads a program that computes 5 * 3 by repeated addition (uses LDA, STA, ADD, SUB, JZ, JMP, HLT).

In [154]:
cpu_testbench = '''
`timescale 1ns/1ps
module tb_cpu();
    reg clk = 0, rst_n = 0;
    wire [7:0] acc_out;
    wire halted;
    integer i;

    cpu dut (.clk(clk), .rst_n(rst_n), .acc_out(acc_out), .halted(halted));

    always #5 clk = ~clk;

    initial begin
        for (i = 0; i < 16; i = i + 1) dut.mem[i] = 8'd0;
        // Program: M[12] = M[10] * M[11] by repeated addition
        dut.mem[0]  = {3'b000, 5'd12}; // LDA 12
        dut.mem[1]  = {3'b010, 5'd10}; // ADD 10
        dut.mem[2]  = {3'b001, 5'd12}; // STA 12
        dut.mem[3]  = {3'b000, 5'd11}; // LDA 11
        dut.mem[4]  = {3'b011, 5'd13}; // SUB 13
        dut.mem[5]  = {3'b001, 5'd11}; // STA 11
        dut.mem[6]  = {3'b110, 5'd8};  // JZ 8
        dut.mem[7]  = {3'b101, 5'd0};  // JMP 0
        dut.mem[8]  = {3'b000, 5'd12}; // LDA 12
        dut.mem[9]  = {3'b111, 5'd0};  // HLT
        dut.mem[10] = 8'd5;            // A
        dut.mem[11] = 8'd3;            // B (loop counter)
        dut.mem[13] = 8'd1;            // constant 1

        #12 rst_n = 1;
        wait (halted);
        @(posedge clk);
        if (acc_out == 8'd15 && dut.mem[12] == 8'd15)
            $display("CPU test passed: 5 * 3 = %0d", acc_out);
        else
            $display("CPU test failed: expected 15, got ACC=%0d M[12]=%0d", acc_out, dut.mem[12]);
        $finish;
    end

    initial begin
        #10000 $display("CPU test failed: timeout"); $finish;
    end
endmodule
'''
with open("cpu/cpu_tb.v", "w") as f:
    f.write(cpu_testbench)

In [155]:
!cd cpu/ && iverilog -g2012 -o cpu.vvp cpu.v cpu_tb.v && vvp cpu.vvp

cpu.v:14: error: 'halted' has already been declared in this scope.
cpu.v:6:      : It was declared here as a net.


In [156]:
feedback_prompt = '''
The design does not compile with iverilog:
cpu.v:14: error: 'halted' has already been declared in this scope.
cpu.v:6:      : It was declared here as a net.
Please fix this. Only give the Verilog module.
'''

cpu_messages.append({"role":"user","content":feedback_prompt})
completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=cpu_messages,
  extra_body={"chat_template_kwargs": {"enable_thinking": False}},
  max_tokens=4096,
  stream=False
)
cpu_messages.append({"role":"assistant","content":completion.choices[0].message.content})
print(completion.choices[0].message.content)

```verilog
module cpu (
    input        clk,
    input        rst_n,      // active-low asynchronous reset
    output [7:0] acc_out,
    output       halted
);

    // 5-bit PC for 32-byte memory
    reg [4:0]  PC;
    // 8-bit accumulator
    reg [7:0]  ACC;
    // Halt flag
    reg        halted_reg;
    // FSM state: 0 = FETCH, 1 = EXECUTE
    reg [1:0]  state;
    // Instruction register
    reg [7:0]  IR;
    // Internal memory (32x8)
    reg [7:0]  mem [0:31];

    // Output assignments
    assign acc_out = ACC;
    assign halted  = halted_reg;

    // State encoding
    localparam FETCH = 2'b00;
    localparam EXEC  = 2'b01;

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            // Synchronous reset (async due to negedge)
            PC       <= 5'h0;
            ACC      <= 8'h0;
            halted_reg <= 1'b0;
            state    <= FETCH;
            IR       <= 8'h0;
        end else begin
            if (halted_reg) begin
                /

In [157]:
output_verilog_code = '''
module cpu (
    input        clk,
    input        rst_n,      // active-low asynchronous reset
    output [7:0] acc_out,
    output       halted
);

    // 5-bit PC for 32-byte memory
    reg [4:0]  PC;
    // 8-bit accumulator
    reg [7:0]  ACC;
    // Halt flag
    reg        halted_reg;
    // FSM state: 0 = FETCH, 1 = EXECUTE
    reg [1:0]  state;
    // Instruction register
    reg [7:0]  IR;
    // Internal memory (32x8)
    reg [7:0]  mem [0:31];

    // Output assignments
    assign acc_out = ACC;
    assign halted  = halted_reg;

    // State encoding
    localparam FETCH = 2'b00;
    localparam EXEC  = 2'b01;

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            // Synchronous reset (async due to negedge)
            PC       <= 5'h0;
            ACC      <= 8'h0;
            halted_reg <= 1'b0;
            state    <= FETCH;
            IR       <= 8'h0;
        end else begin
            if (halted_reg) begin
                // Remain halted; do nothing
            end else begin
                case (state)
                    FETCH: begin
                        // Fetch instruction from memory[PC]
                        IR <= mem[PC];
                        state <= EXEC;
                    end
                    EXEC: begin
                        // Execute instruction based on opcode IR[7:5]
                        case (IR[7:5])
                            3'b000: // LDA: ACC <- mem[addr]
                                ACC <= mem[IR[4:0]];
                            3'b001: // STA: mem[addr] <- ACC
                                mem[IR[4:0]] <= ACC;
                            3'b010: // ADD: ACC <- ACC + mem[addr]
                                ACC <= ACC + mem[IR[4:0]];
                            3'b011: // SUB: ACC <- ACC - mem[addr]
                                ACC <= ACC - mem[IR[4:0]];
                            3'b100: // AND: ACC <- ACC & mem[addr]
                                ACC <= ACC & mem[IR[4:0]];
                            3'b101: // JMP: PC <- addr
                                PC <= IR[4:0];
                            3'b110: // JZ: if ACC==0 then PC <- addr else PC <- PC+1
                                if (ACC == 8'h0)
                                    PC <= IR[4:0];
                                else
                                    PC <= PC + 1;
                            3'b111: // HLT: set halted flag
                                halted_reg <= 1'b1;
                        endcase

                        // Default PC increment for non-branch, non-halt instructions
                        if (IR[7:5] == 3'b000 || IR[7:5] == 3'b001 ||
                            IR[7:5] == 3'b010 || IR[7:5] == 3'b011 ||
                            IR[7:5] == 3'b100) begin
                            PC <= PC + 1;
                        end

                        // Return to fetch state
                        state <= FETCH;
                    end
                endcase
            end
        end
    end
endmodule
'''
filename = "cpu/cpu.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [158]:
!cd cpu/ && iverilog -g2012 -o cpu.vvp cpu.v cpu_tb.v && vvp cpu.vvp

CPU test passed: 5 * 3 = 15
cpu_tb.v:37: $finish called at 515000 (1ps)


Check the 1000-standard-cell budget with yosys.

In [161]:
!apt-get install -y yosys > /dev/null

In [163]:
!cd cpu/ && yosys -p "read_verilog cpu.v; synth -top cpu; abc -g AND,NAND,OR,NOR,XOR,XNOR,MUX; stat" | grep -A 20 "Number of cells" | tail -21


3. Executing ABC pass (technology mapping using ABC).
--
   Number of cells:               1601
     $_AND_                        594
     $_DFFE_PN0P_                   21
     $_DFFE_PP_                    256
     $_DFF_PN0_                      2
     $_DFF_PN1_                      1
     $_MUX_                         15
     $_NAND_                       592
     $_NOR_                         64
     $_NOT_                         15
     $_OR_                          23
     $_XNOR_                        16
     $_XOR_                          2

Warnings: 1 unique messages, 1 total
End of script. Logfile hash: 62b06bca78, CPU: user 1.73s system 0.01s, MEM: 19.86 MB peak
Yosys 0.33 (git sha1 2584903a060)
Time spent: 38% 2x abc (1 sec), 20% 22x opt_expr (0 sec), ...


In [164]:
feedback_prompt = '''
The design now passes my testbench, but synthesizing it with yosys gives 1601 cells, which is over our 1000-cell budget.
Reduce the memory to 16 bytes: use a 4-bit PC and a 4-bit address (instruction bits [3:0]; bit 4 is unused). Keep everything else the same. Only give the Verilog module.
'''

cpu_messages.append({"role":"user","content":feedback_prompt})
completion = client.chat.completions.create(
  model="nvidia/nemotron-3-super-120b-a12b",
  messages=cpu_messages,
  extra_body={"chat_template_kwargs": {"enable_thinking": False}},
  max_tokens=4096,
  stream=False
)
cpu_messages.append({"role":"assistant","content":completion.choices[0].message.content})
print(completion.choices[0].message.content)

```verilog
module cpu (
    input        clk,
    input        rst_n,      // active-low asynchronous reset
    output [7:0] acc_out,
    output       halted
);

    // 4-bit PC for 16-byte memory
    reg [3:0]  PC;
    // 8-bit accumulator
    reg [7:0]  ACC;
    // Halt flag
    reg        halted_reg;
    // FSM state: 0 = FETCH, 1 = EXECUTE
    reg [1:0]  state;
    // Instruction register
    reg [7:0]  IR;
    // Internal memory (16x8)
    reg [7:0]  mem [0:15];

    // Output assignments
    assign acc_out = ACC;
    assign halted  = halted_reg;

    // State encoding
    localparam FETCH = 2'b00;
    localparam EXEC  = 2'b01;

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            // Synchronous reset (async due to negedge)
            PC       <= 4'h0;
            ACC      <= 8'h0;
            halted_reg <= 1'b0;
            state    <= FETCH;
            IR       <= 8'h0;
        end else begin
            if (halted_reg) begin
                /

In [165]:
output_verilog_code = '''
module cpu (
    input        clk,
    input        rst_n,      // active-low asynchronous reset
    output [7:0] acc_out,
    output       halted
);

    // 4-bit PC for 16-byte memory
    reg [3:0]  PC;
    // 8-bit accumulator
    reg [7:0]  ACC;
    // Halt flag
    reg        halted_reg;
    // FSM state: 0 = FETCH, 1 = EXECUTE
    reg [1:0]  state;
    // Instruction register
    reg [7:0]  IR;
    // Internal memory (16x8)
    reg [7:0]  mem [0:15];

    // Output assignments
    assign acc_out = ACC;
    assign halted  = halted_reg;

    // State encoding
    localparam FETCH = 2'b00;
    localparam EXEC  = 2'b01;

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            // Synchronous reset (async due to negedge)
            PC       <= 4'h0;
            ACC      <= 8'h0;
            halted_reg <= 1'b0;
            state    <= FETCH;
            IR       <= 8'h0;
        end else begin
            if (halted_reg) begin
                // Remain halted; do nothing
            end else begin
                case (state)
                    FETCH: begin
                        // Fetch instruction from memory[PC]
                        IR <= mem[PC];
                        state <= EXEC;
                    end
                    EXEC: begin
                        // Execute instruction based on opcode IR[7:5]
                        case (IR[7:5])
                            3'b000: // LDA: ACC <- mem[addr]
                                ACC <= mem[IR[3:0]];
                            3'b001: // STA: mem[addr] <- ACC
                                mem[IR[3:0]] <= ACC;
                            3'b010: // ADD: ACC <- ACC + mem[addr]
                                ACC <= ACC + mem[IR[3:0]];
                            3'b011: // SUB: ACC <- ACC - mem[addr]
                                ACC <= ACC - mem[IR[3:0]];
                            3'b100: // AND: ACC <- ACC & mem[addr]
                                ACC <= ACC & mem[IR[3:0]];
                            3'b101: // JMP: PC <- addr
                                PC <= IR[3:0];
                            3'b110: // JZ: if ACC==0 then PC <- addr else PC <- PC+1
                                if (ACC == 8'h0)
                                    PC <= IR[3:0];
                                else
                                    PC <= PC + 1;
                            3'b111: // HLT: set halted flag
                                halted_reg <= 1'b1;
                        endcase

                        // Default PC increment for non-branch, non-halt instructions
                        if (IR[7:5] == 3'b000 || IR[7:5] == 3'b001 ||
                            IR[7:5] == 3'b010 || IR[7:5] == 3'b011 ||
                            IR[7:5] == 3'b100) begin
                            PC <= PC + 1;
                        end

                        // Return to fetch state
                        state <= FETCH;
                    end
                endcase
            end
        end
    end
endmodule
'''
filename = "cpu/cpu.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [166]:
!cd cpu/ && iverilog -g2012 -o cpu.vvp cpu.v cpu_tb.v && vvp cpu.vvp

CPU test passed: 5 * 3 = 15
cpu_tb.v:37: $finish called at 515000 (1ps)


In [167]:
!cd cpu/ && yosys -p "read_verilog cpu.v; synth -top cpu; abc -g AND,NAND,OR,NOR,XOR,XNOR,MUX; stat" | grep -A 20 "Number of cells" | tail -21


3. Executing ABC pass (technology mapping using ABC).
--
   Number of cells:                838
     $_AND_                        296
     $_DFFE_PN0P_                   19
     $_DFFE_PP_                    128
     $_DFF_PN0_                      2
     $_DFF_PN1_                      1
     $_MUX_                         15
     $_NAND_                       303
     $_NOR_                         23
     $_NOT_                         17
     $_OR_                          17
     $_XNOR_                        13
     $_XOR_                          4

Warnings: 1 unique messages, 1 total
End of script. Logfile hash: 9177a51f9d, CPU: user 1.19s system 0.02s, MEM: 17.12 MB peak
Yosys 0.33 (git sha1 2584903a060)
Time spent: 30% 2x abc (0 sec), 23% 22x opt_expr (0 sec), ...


In [168]:
!zip -r HW_ChipChat_outputs.zip binary_to_bcd sequence_detector shift_register abro_state_machine dice_roller lfsr sequence_generator traffic_light cpu
from google.colab import files
files.download("HW_ChipChat_outputs.zip")

  adding: binary_to_bcd/ (stored 0%)
  adding: binary_to_bcd/my_design.vcd (deflated 70%)
  adding: binary_to_bcd/binary_to_bcd_tb.v (deflated 55%)
  adding: binary_to_bcd/binary_to_bcd.vvp (deflated 70%)
  adding: binary_to_bcd/binary_to_bcd.v (deflated 46%)
  adding: sequence_detector/ (stored 0%)
  adding: sequence_detector/sequence_detector_tb.v (deflated 70%)
  adding: sequence_detector/sequence_detector.vvp (deflated 87%)
  adding: sequence_detector/sequence_detector.v (deflated 77%)
  adding: shift_register/ (stored 0%)
  adding: shift_register/shift_register.v (deflated 57%)
  adding: shift_register/shift_register.vvp (deflated 75%)
  adding: shift_register/shift_register_tb.v (deflated 64%)
  adding: abro_state_machine/ (stored 0%)
  adding: abro_state_machine/my_design.vcd (deflated 66%)
  adding: abro_state_machine/abro_state_machine_tb.v (deflated 68%)
  adding: abro_state_machine/abro_state_machine.vvp (deflated 81%)
  adding: abro_state_machine/abro_state_machine.v (defla

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>